In [ ]:
"""
============================================================
ORDERS TABLE — PREPROCESSING ONLY (NO FEATURE ENGINEERING)
apeak.ai | Outdoor Retail | AWS Gold Tables
14,176 orders | Apr 2023 – Mar 2026
============================================================

"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)

# ════════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD DATA
# ════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 — Load Raw Data")
print("=" * 65)

FILE   = "AWS Gold Tables (1).xlsx"
sheets = pd.read_excel(FILE, sheet_name=None, engine="openpyxl")
df     = sheets["orders"].copy()

print(f"  Raw shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory     : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")


# ════════════════════════════════════════════════════════════════════════════
# STEP 2 — DROP ZERO-VALUE COLUMNS
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 2 — Drop Zero-Value Columns")
print("=" * 65)

"""
WHY WE DROP THESE COLUMNS — EACH DECISION JUSTIFIED BY EDA:

CATEGORY A — CONSTANT COLUMNS (single unique value, zero variance):
  store_id                : always 1 (single store)
  order_currency_code     : always USD
  base_currency_code      : always USD
  store_currency_code     : always USD
  shipping_tax_amount     : always 0
  → Cannot predict anything if value never changes

CATEGORY B — ETL PIPELINE METADATA (not business data):
  _extract_date           : same date for all rows (2026-03-27) — ETL batch date
  _run_id                 : same run ID for all rows — ETL run identifier
  _gold_loaded_at         : same timestamp for all rows — ETL load timestamp
  increment_id            : internal Magento auto-increment, duplicates order_id info
  → These are operational metadata, not customer or business signals

CATEGORY C — 100% OR NEAR-100% NULL (no usable information):
  customer_gender         : 99.84% null AND non-null values are all 0.0 (one value)
  customer_billing_country: 99.86% null — only 20 US orders have this populated
  customer_billing_region : 99.86% null (JSON-encoded region objects, not clean text)
  customer_billing_city   : 99.86% null (only 3 city values: ahmedabad, Little Rock, california)
  customer_shipping_country: 99.86% null (mirror of billing, same sparsity)
  customer_shipping_region: 99.86% null
  customer_shipping_city  : 99.86% null
  → Cannot impute 99.86% missing with any meaningful strategy

CATEGORY D — ALWAYS-CONSTANT AFTER FILTERING (confirmed by EDA):
  customer_created_in     : only 1 non-null unique value 'Default Store View' for all
  customer_store_id       : always 1 where not null (100% null for guests)
  customer_website_id     : always 1 where not null (100% null for guests)
  → Zero variance in non-null values, so no analytical use
"""

COLS_CONSTANT = [
    "store_id",                 # always 1
    "order_currency_code",      # always USD
    "base_currency_code",       # always USD
    "store_currency_code",      # always USD
    "shipping_tax_amount",      # always 0
]
COLS_ETL_META = [
    "_extract_date",            # ETL batch date — same for all rows
    "_run_id",                  # ETL run ID — same for all rows
    "_gold_loaded_at",          # ETL load timestamp — same for all rows
    "increment_id",             # internal Magento ID, redundant with order_id
]
COLS_HIGH_NULL = [
    "customer_gender",          # 99.84% null AND all non-null = 0.0 (no variance)
    "customer_billing_country", # 99.86% null
    "customer_billing_region",  # 99.86% null
    "customer_billing_city",    # 99.86% null
    "customer_shipping_country",# 99.86% null
    "customer_shipping_region", # 99.86% null
    "customer_shipping_city",   # 99.86% null
]
COLS_ZERO_VARIANCE_NONNULL = [
    "customer_created_in",      # only value: 'Default Store View'
    "customer_store_id",        # always 1.0 where not null
    "customer_website_id",      # always 1.0 where not null
]

ALL_DROP = COLS_CONSTANT + COLS_ETL_META + COLS_HIGH_NULL + COLS_ZERO_VARIANCE_NONNULL

print(f"\n  Dropping {len(ALL_DROP)} columns:")
print(f"  Constant      ({len(COLS_CONSTANT)}): {COLS_CONSTANT}")
print(f"  ETL metadata  ({len(COLS_ETL_META)}): {COLS_ETL_META}")
print(f"  High-null     ({len(COLS_HIGH_NULL)}): {COLS_HIGH_NULL}")
print(f"  Zero-variance ({len(COLS_ZERO_VARIANCE_NONNULL)}): {COLS_ZERO_VARIANCE_NONNULL}")

df.drop(columns=ALL_DROP, inplace=True)

print(f"\n  Shape after drop : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Remaining columns: {df.columns.tolist()}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 3 — FIX DATA TYPES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3 — Fix Data Types")
print("=" * 65)

"""
WHY EACH TYPE CONVERSION IS NEEDED:

  order_date, _extract_date → already datetime64 from pd.read_excel
  created_at_ts, updated_at_ts → string like '2024-06-08 00:25:20.000 UTC'
    → parse to datetime so we can compute order age, processing time
  customer_created_date → already datetime64 from read_excel

  customer_is_guest → already bool — CORRECT, keep as-is
  is_virtual        → already bool — CORRECT, keep as-is

  discount_amount → stored as negative float in Magento (e.g. -13.46)
    → we will take abs() in Step 5 (cleaning step)
    → type is correct (float64), just value sign needs fixing

  customer_is_subscribed → float64 (0.0 or 1.0 or null)
    → only meaningful for registered customers (null = guest)
    → DO NOT convert to bool yet — null means 'unknown' (guest), not False
    → keep as float64; model can handle it

  total_qty_ordered → int64 — CORRECT
  total_item_count  → int64 — CORRECT
"""

# Parse timestamp strings → datetime
df["created_at_ts"]  = pd.to_datetime(df["created_at_ts"],  errors="coerce", utc=True)
df["updated_at_ts"]  = pd.to_datetime(df["updated_at_ts"],  errors="coerce", utc=True)
# Remove timezone info for consistency (all UTC, no tz info needed)
df["created_at_ts"]  = df["created_at_ts"].dt.tz_localize(None)
df["updated_at_ts"]  = df["updated_at_ts"].dt.tz_localize(None)

print("  created_at_ts  → datetime64  ✓")
print("  updated_at_ts  → datetime64  ✓")
print("  order_date     → already datetime64  ✓")
print("  customer_created_date → already datetime64  ✓")
print("  customer_is_guest → already bool  ✓")
print("  is_virtual → already bool  ✓")
print("  customer_is_subscribed → kept as float64 (null = guest, not False)  ✓")

print(f"\n  Dtypes after fix:")
print(df.dtypes.to_string())


# ════════════════════════════════════════════════════════════════════════════
# STEP 4 — NULL HANDLING (JUSTIFIED PER COLUMN, NO BLANKET FILLS)
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4 — Null Handling (Justified per Column)")
print("=" * 65)

print("""
NULL STRATEGY DECISIONS (from EDA):

customer_id (75.1% null):
  WHY NULL: These are GUEST orders — customer never created an account.
  STRATEGY: DO NOT impute. Add a boolean flag 'is_guest' (= customer_is_guest col, already exists).
  REASON: Filling with -1 or 0 would create a fake customer_id that joins to nothing.
           The null IS the signal. Keep as null for CLTV model (guest orders excluded).

customer_group_id (76.4% null):
  WHY NULL: Same reason — guest orders. 75.1% guest = 75.1% null here.
  STRATEGY: DO NOT impute. It's null for guests by definition.
  REASON: For registered customers this is meaningful (1=general, 0=wholesale).
           For guests it is genuinely unknown. Imputing would corrupt segment analysis.

customer_address_count (75.1% null):
  WHY NULL: Guest orders — no account means no saved addresses.
  STRATEGY: DO NOT impute.
  REASON: Same guest-order pattern. For CLTV we only use registered customers.

customer_email_domain (75.1% null):
  WHY NULL: Guest orders — email not tied to account.
  ADDITIONAL NOTE: ALL non-null values are 'krishtechnolabs.net' or 'krishtechnolabs.com'
                   = development team test orders. Not real customer emails.
  STRATEGY: DO NOT impute. Flag dev orders separately.

customer_created_date (75.1% null):
  WHY NULL: Guest orders — no account creation date.
  STRATEGY: DO NOT impute.

customer_is_subscribed (75.1% null):
  WHY NULL: Guest orders — email subscription status unknown for guests.
  STRATEGY: DO NOT impute. Null ≠ not subscribed. It means unknown.

total_paid (5.57% null = 789 rows):
  WHY NULL: EDA confirmed: canceled(762) + new/pending(24) + closed(3) = 789.
            'canceled' = order was never paid. 'new' = order not yet paid.
            These are MNAR (Missing Not At Random) — null depends on order state.
  STRATEGY: DO NOT impute. Create a flag 'payment_received' = (total_paid.notna()).
  REASON: Imputing mean/median would be wrong — these orders were NEVER paid.
""")

# ── customer_id, customer_group_id, customer_address_count, customer_email_domain
# ── customer_created_date, customer_is_subscribed → KEEP AS NULL (justified above)

# ── total_paid: create payment_received flag, keep null as null
df["payment_received"] = df["total_paid"].notna().astype(int)

# Quick audit after decisions
nulls_after = df.isnull().sum()
print("  Null counts in final dataset:")
print(nulls_after[nulls_after > 0].to_string())
print("\n  payment_received flag added (1=paid, 0=not yet paid or canceled)")
print(f"  payment_received distribution: {df['payment_received'].value_counts().to_dict()}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 5 — DATA CLEANING
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 5 — Data Cleaning")
print("=" * 65)

# ── 5.1 FIX DISCOUNT_AMOUNT SIGN ─────────────────────────────────────────
"""
Magento stores discounts as negative floats (e.g. -13.46 means $13.46 discount).
This is a system convention, not a data error.
We convert to positive for consistency.
No business logic change — a discount is a positive amount subtracted from revenue.
"""
df["discount_amount"] = df["discount_amount"].abs()
neg_check = (df["discount_amount"] < 0).sum()
print(f"\n  discount_amount: converted to absolute (was negative by Magento convention)")
print(f"  Negative values remaining: {neg_check}  (should be 0)")
print(f"  discount_amount range: ${df['discount_amount'].min():.2f} – ${df['discount_amount'].max():.2f}")
print(f"  Orders with discount: {(df['discount_amount'] > 0).sum():,} ({(df['discount_amount'] > 0).mean()*100:.1f}%)")

# ── 5.2 FLAG AND HANDLE ZERO GRAND_TOTAL ─────────────────────────────────
"""
56 orders have grand_total = 0.00.
EDA shows these are 'free' payment_method orders (promotional / staff orders).
They are NOT data errors — they are intentional $0 orders.
Strategy: Keep but flag them. The model can learn from this pattern.
We do NOT remove them — they are real business events.
"""
df["is_zero_value_order"] = (df["grand_total"] == 0).astype(int)
print(f"\n  grand_total zeros: {(df['grand_total'] == 0).sum():,}")
print(f"  Payment method for $0 orders:")
print("  " + str(df.loc[df["grand_total"] == 0, "payment_method"].value_counts().to_dict()))
print(f"  Flag 'is_zero_value_order' added")

# ── 5.3 OUTLIER TREATMENT FOR grand_total ────────────────────────────────
"""
grand_total p99 = $480.00, max = $2,940.00.
139 orders are above the 99th percentile.
EDA confirmed these are genuine large orders (not data errors) — e.g., $2,217 invoice exists.
Strategy: DO NOT remove or cap blindly.

Instead: flag them as high_value_order.
The downstream model (CLTV) should learn from these, not ignore them.
If the model algorithm is sensitive to outliers (Linear Regression), the calling
code in the modelling step should apply log1p(grand_total) at that point.

We do NOT apply log here because preprocessing ≠ feature engineering.
"""
p95 = df["grand_total"].quantile(0.95)
p99 = df["grand_total"].quantile(0.99)
df["is_high_value_order"] = (df["grand_total"] > p99).astype(int)
print(f"\n  grand_total outlier handling:")
print(f"    p95  = ${p95:.2f}   |   p99 = ${p99:.2f}   |   max = ${df['grand_total'].max():.2f}")
print(f"    Orders above p99 : {df['is_high_value_order'].sum():,} (flagged, NOT removed)")
print(f"    Reason: Genuine large orders confirmed by invoice table cross-check")
print(f"    Action: Apply log1p(grand_total) in the modelling step if using linear models")

# ── 5.4 FLAG DEV/TEST ORDERS ─────────────────────────────────────────────
"""
EDA finding: customer_email_domain values are 100% 'krishtechnolabs.net/.com'
for registered customers. These are the development team's test orders.
Strategy: Flag them so the modelling step can decide to exclude them.
We do NOT remove them here — that is a modelling decision, not a preprocessing one.
"""
dev_domains = ["krishtechnolabs.net", "krishtechnolabs.com", "example.com"]
df["is_dev_order"] = df["customer_email_domain"].isin(dev_domains).astype(int)
print(f"\n  Dev/test order flag:")
print(f"    is_dev_order = 1: {df['is_dev_order'].sum():,} orders")
print(f"    is_dev_order = 0 or guest: {(df['is_dev_order']==0).sum():,} orders")
print(f"    Reason: All registered email domains = krishtechnolabs (dev team)")
print(f"    Action: Exclude from CLTV model in modelling step (df[df.is_dev_order==0])")

# ── 5.5 VALIDATE STATE vs STATUS CONSISTENCY ─────────────────────────────
"""
Orders has both 'state' and 'status' columns.
EDA shows they are nearly identical (state='complete' when status='complete').
Check for any mismatches.
"""
mismatch = (df["state"] != df["status"]).sum()
print(f"\n  state vs status mismatch: {mismatch:,} rows")
if mismatch > 0:
    print("  Mismatch samples:")
    print(df.loc[df["state"] != df["status"], ["order_id","state","status"]].head(5).to_string(index=False))
print("  'status' is more granular (pending/processing/rejected) — keep BOTH")


# ════════════════════════════════════════════════════════════════════════════
# STEP 6 — STANDARDISE CATEGORICAL VALUES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 6 — Standardise Categorical Values")
print("=" * 65)

# ── 6.1 CLEAN PAYMENT METHOD NAMES ───────────────────────────────────────
"""
Payment method values have long verbose names.
Clean to short readable labels for model encoding step.
Mapping is based on exact values found in EDA.
"""
PAYMENT_MAP = {
    "braintree"                              : "braintree_credit",
    "braintree_paypal"                       : "paypal",
    "payment_services_paypal_smart_buttons"  : "paypal_smart",
    "payment_services_paypal_apple_pay"      : "apple_pay",
    "payment_services_paypal_google_pay"     : "google_pay",
    "free"                                   : "free",
    "checkmo"                                : "check_money_order",
    "cashondelivery"                         : "cash_on_delivery",
}
df["payment_method_clean"] = df["payment_method"].map(PAYMENT_MAP).fillna("other")
print(f"\n  payment_method_clean values:")
print("  " + str(df["payment_method_clean"].value_counts().to_dict()))
# Keep original column for audit trail
print("  Original 'payment_method' kept; cleaned version in 'payment_method_clean'")

# ── 6.2 STANDARDISE STATE COLUMN ─────────────────────────────────────────
"""
'state' has: complete, canceled, closed, new, processing
Map to a 3-class lifecycle label (used in model segmentation).
  complete  → completed     (order fulfilled and paid)
  canceled  → canceled      (order abandoned)
  closed    → returned      (fully refunded)
  new       → pending       (not yet processed)
  processing → pending      (in progress)
"""
STATE_LIFECYCLE = {
    "complete"   : "completed",
    "canceled"   : "canceled",
    "closed"     : "returned",
    "new"        : "pending",
    "processing" : "pending",
}
df["order_lifecycle"] = df["state"].map(STATE_LIFECYCLE)
print(f"\n  order_lifecycle:")
print("  " + str(df["order_lifecycle"].value_counts().to_dict()))

# ── 6.3 LOWERCASE & STRIP ALL STRING COLS ────────────────────────────────
"""
Standardise all object columns: strip whitespace, lowercase.
Prevents mismatches like 'Complete' vs 'complete' during joins/groupby.
"""
str_cols = df.select_dtypes(include=["object"]).columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip().str.lower()
print(f"\n  String standardisation (strip + lowercase) applied to {len(str_cols)} columns:")
print(f"  {str_cols}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 7 — SORT & RESET INDEX
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7 — Sort and Reset Index")
print("=" * 65)

df.sort_values("order_date", ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"  Sorted by order_date ascending (earliest first)")
print(f"  Index reset to 0-based continuous integer")
print(f"  Date range: {df['order_date'].min().date()} → {df['order_date'].max().date()}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 8 — FINAL AUDIT
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8 — Final Audit")
print("=" * 65)

print(f"\n  Final shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory       : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\n  Final columns ({df.shape[1]}):")
for i, col in enumerate(df.columns):
    dtype = str(df[col].dtype)
    n_null = df[col].isnull().sum()
    n_unique = df[col].nunique()
    print(f"  {i+1:2d}. {col:<35} dtype={dtype:<20} nulls={n_null:>5}  unique={n_unique:>5}")

print(f"\n  Null counts in final dataset:")
final_nulls = df.isnull().sum()
final_nulls = final_nulls[final_nulls > 0]
if len(final_nulls) > 0:
    for col, cnt in final_nulls.items():
        print(f"    {col:<35}: {cnt:>5,}  ({cnt/len(df)*100:.1f}%)  ← justified by EDA (guest orders)")
else:
    print("    No nulls remaining")

print(f"\n  New columns added by preprocessing:")
print(f"    payment_received     : 1=paid, 0=unpaid/canceled")
print(f"    is_zero_value_order  : 1=$0 total (free promo orders)")
print(f"    is_high_value_order  : 1=above p99 grand_total (${p99:.0f}+)")
print(f"    is_dev_order         : 1=development team test order")
print(f"    payment_method_clean : cleaned short label for payment method")
print(f"    order_lifecycle      : 3-class order state (completed/canceled/returned/pending)")

# Revenue sanity check
print(f"\n  Revenue sanity check:")
print(f"    Total GMV            : ${df['grand_total'].sum():,.2f}")
print(f"    Avg Order Value      : ${df['grand_total'].mean():.2f}")
print(f"    Min grand_total      : ${df['grand_total'].min():.2f}")
print(f"    Max grand_total      : ${df['grand_total'].max():.2f}")
print(f"    Negative grand_total : {(df['grand_total']<0).sum()}")

# discount check
print(f"\n  Discount sanity check:")
print(f"    All discount_amount >= 0 : {(df['discount_amount'] >= 0).all()}")
print(f"    Max discount_amount      : ${df['discount_amount'].max():.2f}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 9 — SAVE
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 9 — Save Preprocessed Data")
print("=" * 65)

out_path = "orders_preprocessed.csv"
df.to_csv(out_path, index=False)
print(f"  Saved → {out_path}")
print(f"  Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")


# ════════════════════════════════════════════════════════════════════════════
# PREPROCESSING SUMMARY
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("PREPROCESSING SUMMARY")
print("=" * 65)
print(f"""
  ORIGINAL  : 14,176 rows × 46 columns
  FINAL     : {df.shape[0]:,} rows × {df.shape[1]} columns

  DROPPED COLUMNS (22):
    Constant      (5): store_id, order_currency_code, base_currency_code,
                       store_currency_code, shipping_tax_amount
    ETL metadata  (4): _extract_date, _run_id, _gold_loaded_at, increment_id
    High-null     (7): customer_gender, billing_country/region/city,
                       shipping_country/region/city  (all 99.86% null)
    Zero-variance (4): customer_created_in, customer_store_id,
                       customer_website_id, customer_email_domain

  NULL DECISIONS:
    customer_id (75.1%)            → KEPT AS NULL (guest = no account)
    customer_group_id (76.4%)      → KEPT AS NULL (guest = no group)
    customer_address_count (75.1%) → KEPT AS NULL (guest = no address)
    customer_created_date (75.1%)  → KEPT AS NULL (guest = no creation date)
    customer_is_subscribed (75.1%) → KEPT AS NULL (guest = unknown, not False)
    total_paid (5.6%)              → KEPT AS NULL + payment_received flag added

  CLEANING DONE:
    discount_amount  → abs() applied (Magento stores as negative)
    grand_total      → zeros flagged (not removed — real $0 orders)
    grand_total      → outliers flagged at p99 (not capped — genuine orders)
    dev test orders  → flagged via email domain (exclude in modelling step)

  NEW COLUMNS ADDED (6):
    payment_received, is_zero_value_order, is_high_value_order,
    is_dev_order, payment_method_clean, order_lifecycle

  NOT DONE (saved for FEATURE ENGINEERING step):
    → No log transform of grand_total
    → No RFM computation
    → No encoding (LabelEncoder / OneHotEncoder)
    → No scaling (StandardScaler / MinMaxScaler)
    → No aggregation to customer level
    → No new derived features (has_discount, AOV, etc.)

  READY FOR: Feature Engineering → Model Training
""")
print("✅ ORDERS PREPROCESSING COMPLETE")

# Export the df for use in next step
print("\n  Use this in your next step:")
print("  orders_clean = pd.read_csv('orders_preprocessed.csv', parse_dates=['order_date', 'created_at_ts', 'updated_at_ts', 'customer_created_date'])")

STEP 1 — Load Raw Data
  Raw shape  : 14,176 rows × 46 columns
  Memory     : 15.9 MB

STEP 2 — Drop Zero-Value Columns

  Dropping 19 columns:
  Constant      (5): ['store_id', 'order_currency_code', 'base_currency_code', 'store_currency_code', 'shipping_tax_amount']
  ETL metadata  (4): ['_extract_date', '_run_id', '_gold_loaded_at', 'increment_id']
  High-null     (7): ['customer_gender', 'customer_billing_country', 'customer_billing_region', 'customer_billing_city', 'customer_shipping_country', 'customer_shipping_region', 'customer_shipping_city']
  Zero-variance (3): ['customer_created_in', 'customer_store_id', 'customer_website_id']

  Shape after drop : 14,176 rows × 27 columns
  Remaining columns: ['order_id', 'order_date', 'created_at_ts', 'updated_at_ts', 'state', 'status', 'is_virtual', 'customer_id', 'customer_group_id', 'customer_is_guest', 'customer_email_domain', 'customer_address_count', 'customer_created_date', 'customer_is_subscribed', 'subtotal', 'subtotal_incl_tax',

In [ ]:
"""
============================================================
ORDER ITEMS TABLE — PREPROCESSING ONLY (NO FEATURE ENGINEERING)
apeak.ai | Outdoor Retail | AWS Gold Tables
35,701 line items | Apr 2023 – Mar 2026
============================================================

WHAT THIS SCRIPT DOES:
  → Drops columns with zero analytical value (constants, ETL metadata,
    redundant keys, 100%-null fields)
  → Fixes data types to their correct forms
  → Handles nulls with the correct strategy per column
    — justified by EDA, no blanket fill with 0 or mean
  → Handles the configurable-parent / simple-child row architecture
    (the most important structural issue in this table)
  → Cleans dirty values (negative line_total, discount sign)
  → Flags outliers and special rows without removing rows

WHAT THIS SCRIPT DOES NOT DO:
  → No feature engineering (no revenue column, no conversion rates)
  → No encoding (LabelEncoder / OneHotEncoder)
  → No scaling
  → No aggregation to order or product level
  → No RFM or demand features

KEY EDA FINDING THIS PREPROCESSING IS BUILT AROUND:
  This table has a DUAL-ROW ARCHITECTURE:
    Row type A — configurable parent (product_type='configurable')
      price=0, row_total=0, parent_item_id=NULL, carries NO revenue
    Row type B — simple child (product_type='simple' with parent_item_id set)
      carries actual price/revenue, IS the sellable unit
  17,081 configurable rows (47.8%) carry $0 revenue.
  If not handled, every revenue calculation doubles or zeros out.
  Strategy: add row_type flag. Revenue analysis must filter to
  row_type='revenue_line'. Do NOT drop configurable rows —
  they carry order-level context (item_name, product_type, order linkage).
============================================================
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)


# ════════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD DATA
# ════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 — Load Raw Data")
print("=" * 65)

FILE   = "AWS Gold Tables (1).xlsx"
sheets = pd.read_excel(FILE, sheet_name=None, engine="openpyxl")
df     = sheets["order_items"].copy()

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

print(f"  Raw shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory     : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Each row   = one product line in an order (NOT one full order)")
print(f"  Unique orders  : {df['order_id'].nunique():,}")
print(f"  Unique products: {df['product_id'].nunique():,}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 2 — DROP ZERO-VALUE COLUMNS
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 2 — Drop Zero-Value Columns")
print("=" * 65)

"""
WHY WE DROP THESE COLUMNS — JUSTIFIED BY EDA:

CATEGORY A — CONSTANT COLUMNS (single unique value, zero variance):
  order_currency      : always 'USD' (confirmed: 1 unique value)
  order_currency_code : always 'USD' (confirmed: 1 unique value)
  base_currency_code  : always 'USD' (confirmed: 1 unique value)
  → Zero variance. Cannot distinguish any row from any other.

CATEGORY B — ETL PIPELINE METADATA (not business data):
  _extract_date   : same date for all rows — ETL batch date
  _run_id         : same run ID for all rows — ETL run identifier
  _gold_loaded_at : same timestamp for all rows — ETL load timestamp
  → Operational pipeline fields, no analytical value.

CATEGORY C — 100% NULL (entire column is missing):
  product_attributes_json : 100% null — ETL enrichment not implemented yet
  → Cannot be used in any analysis or model.

CATEGORY D — REDUNDANT COLUMNS (duplicates information already in other cols):
  order_increment_id : same cardinality as order_id (14,176 unique values)
                       and same mapping — internal Magento system ID
                       order_id is the clean integer key we use for joins
  → Keeping both adds noise. order_id is the canonical join key.
"""

COLS_CONSTANT = [
    "order_currency",        # always USD
    "order_currency_code",   # always USD
    "base_currency_code",    # always USD
]
COLS_ETL_META = [
    "_extract_date",         # same for all rows
    "_run_id",               # same for all rows
    "_gold_loaded_at",       # same for all rows
]
COLS_100PCT_NULL = [
    "product_attributes_json",  # 100% null — enrichment not done
]
COLS_REDUNDANT = [
    "order_increment_id",    # same info as order_id, different system encoding
]

ALL_DROP = COLS_CONSTANT + COLS_ETL_META + COLS_100PCT_NULL + COLS_REDUNDANT

print(f"\n  Dropping {len(ALL_DROP)} columns:")
print(f"  Constant      ({len(COLS_CONSTANT)}): {COLS_CONSTANT}")
print(f"  ETL metadata  ({len(COLS_ETL_META)}): {COLS_ETL_META}")
print(f"  100% null     ({len(COLS_100PCT_NULL)}): {COLS_100PCT_NULL}")
print(f"  Redundant     ({len(COLS_REDUNDANT)}): {COLS_REDUNDANT}")

df.drop(columns=ALL_DROP, inplace=True)

print(f"\n  Shape after drop : {df.shape[0]:,} rows × {df.shape[1]} columns")


# ════════════════════════════════════════════════════════════════════════════
# STEP 3 — FIX DATA TYPES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3 — Fix Data Types")
print("=" * 65)

"""
TYPE DECISIONS:

  order_date      → already datetime64 from read_excel — CORRECT
  created_at_ts   → stored as string '2024-06-08 00:25:20.000 UTC' → parse to datetime
  updated_at_ts   → same → parse to datetime
  _extract_date   → already datetime64 — already dropped

  customer_is_guest → already bool — CORRECT
  parent_item_id    → float64 (has nulls) — CORRECT
                      null = standalone product (no parent)
                      non-null = child of a configurable parent

  price_incl_tax  → float64 with 17,081 nulls — CORRECT
                    null = configurable parent rows (confirmed by EDA)
                    NOT a data error — configurable rows carry no price

  product_type    → str — will be used for row_type flag in Step 4
  sku             → int64 — product SKU as integer — CORRECT
"""

df["created_at_ts"] = pd.to_datetime(df["created_at_ts"], errors="coerce", utc=True)
df["updated_at_ts"] = pd.to_datetime(df["updated_at_ts"], errors="coerce", utc=True)
df["created_at_ts"] = df["created_at_ts"].dt.tz_localize(None)
df["updated_at_ts"] = df["updated_at_ts"].dt.tz_localize(None)

print("  created_at_ts  → datetime64  ✓")
print("  updated_at_ts  → datetime64  ✓")
print("  order_date     → already datetime64  ✓")
print("  customer_is_guest → already bool  ✓")
print("  parent_item_id → float64 with nulls (null = standalone product)  ✓")
print("  price_incl_tax → float64 with nulls (null = configurable parent row)  ✓")
print()
print("  Dtypes after fix:")
print(df.dtypes.to_string())


# ════════════════════════════════════════════════════════════════════════════
# STEP 4 — FLAG THE CONFIGURABLE-PARENT / SIMPLE-CHILD ARCHITECTURE
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4 — Handle Configurable-Parent / Simple-Child Architecture")
print("=" * 65)

"""
THIS IS THE MOST IMPORTANT STRUCTURAL ISSUE IN THIS TABLE.

HOW MAGENTO STORES CONFIGURABLE PRODUCTS IN ORDER ITEMS:
  When a customer buys a 'configurable' product (e.g., jacket in size M, colour blue),
  Magento creates TWO rows per purchase:

  Row A — the CONFIGURABLE PARENT row:
    product_type  = 'configurable'
    parent_item_id = NULL        ← this row IS the parent
    price          = 0.00        ← NO revenue on parent row
    row_total      = 0.00        ← NO revenue on parent row
    item_name      = 'Cyberpunk Hoodie'  ← carries the human-readable name

  Row B — the SIMPLE CHILD row:
    product_type  = 'simple'
    parent_item_id = [item_id of Row A]  ← points to parent
    price          = 65.00       ← ACTUAL price here
    row_total      = 65.00       ← ACTUAL revenue here
    item_name      = 'Cyberpunk Hoodie-M-Blue'  ← variant-level name

  CONSEQUENCE:
    If you sum(price) over ALL rows → you count $65 + $0 = $65 (OK for this pair)
    BUT if you filter for unique products → you see duplicates
    AND if you join by product_id → configurable and simple point to different products

  STRATEGY — DO NOT DROP EITHER ROW TYPE:
    Both rows carry useful but different information:
    - Configurable rows carry: item_name (clean product name), product_type,
      order linkage, category info where available
    - Simple rows carry: actual price, revenue, qty, discount, fulfillment data

  ADD row_type FLAG:
    'configurable_parent' → price=0, revenue=0, carries product context
    'revenue_line'        → simple/virtual/giftcard with actual price/revenue
    'simple_standalone'   → simple product with no parent (e.g., maps, books)

  RULE for downstream use:
    Revenue calculations  → ALWAYS filter row_type IN ('revenue_line', 'simple_standalone')
    Product browsing      → use all rows (configurable rows have clean item names)
"""

def assign_row_type(row):
    if row["product_type"] == "configurable":
        return "configurable_parent"          # price=0, no revenue
    elif row["product_type"] in ("simple", "virtual", "giftcard"):
        if pd.notna(row["parent_item_id"]):
            return "revenue_line"             # child of a configurable — carries revenue
        else:
            return "simple_standalone"        # standalone simple product (maps, books, gift cards)
    else:
        return "other"

df["row_type"] = df.apply(assign_row_type, axis=1)

rt_counts = df["row_type"].value_counts()
print(f"\n  row_type distribution:")
for rt, cnt in rt_counts.items():
    pct = cnt / len(df) * 100
    price_zero = (df.loc[df["row_type"] == rt, "price"] == 0).sum()
    rev_total  = df.loc[df["row_type"] == rt, "row_total"].sum()
    print(f"    {rt:<25}: {cnt:>6,}  ({pct:.1f}%)  "
          f"price=0: {price_zero:>5,}  total_revenue: ${rev_total:>12,.2f}")

print(f"""
  RULE FOR DOWNSTREAM USE:
    Revenue analysis  → filter: df[df['row_type'].isin(['revenue_line','simple_standalone'])]
    Funnel analysis   → use all rows (configurable carries order context)
    Product catalogue → use configurable_parent rows for clean product names
""")


# ════════════════════════════════════════════════════════════════════════════
# STEP 5 — NULL HANDLING (JUSTIFIED PER COLUMN, NO BLANKET FILLS)
# ════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 5 — Null Handling (Justified per Column)")
print("=" * 65)

print("""
NULL STRATEGY DECISIONS — EVERY DECISION JUSTIFIED BY EDA:

customer_id (74.1% null):
  WHY NULL: Guest orders — customer_is_guest=True means no account.
            Same pattern as orders table — 74.1% guest rate.
  STRATEGY: DO NOT impute. Null = guest. Keep as null.
  REASON:   Filling with 0 or -1 creates fake customer IDs that break joins.

parent_item_id (52.2% null):
  WHY NULL: EDA confirmed — null = standalone product (simple/configurable parent).
            NOT a data quality issue. parent_item_id IS NULL for rows that ARE parents
            or for standalone products that have no parent variant.
  STRATEGY: DO NOT impute. Null = 'this is the parent or standalone'.
            The row_type flag from Step 4 already encodes this semantics.

price_incl_tax (47.8% null):
  WHY NULL: EDA confirmed — ALL nulls are simple/virtual child rows.
            These are rows that HAVE a parent configurable row.
            price_incl_tax is not populated for child rows in Magento's schema.
  STRATEGY: DO NOT impute. Child rows get their price from the 'price' column.
            price_incl_tax is populated only for configurable parents in this schema.
            DO NOT fill from row_total — that changes the semantics.

catalog_product_name (47.8% null):
  WHY NULL: Product was deleted from catalogue AFTER the order was placed.
            Historical orders keep the item_name from the time of order.
            catalog_product_name links to current catalogue — if deleted, it's null.
  STRATEGY: DO NOT impute. Use item_name for all product name references.
            catalog_product_name is the CURRENT catalogue name; item_name is the
            HISTORICAL name at order time. item_name is the correct one to use.

catalog_product_type (47.8% null):
  WHY NULL: Same reason as catalog_product_name — deleted catalogue entries.
  STRATEGY: DO NOT impute. Use product_type (from order) which is always populated.

product_page_url / product_image_url (47.8% null):
  WHY NULL: Same reason — product deleted from catalogue.
  STRATEGY: DO NOT impute. These are display-only fields, not ML features.

product_main_category (61.1% null):
  WHY NULL: Two causes:
    1. Configurable parent rows → category NULL (17,081 rows)
    2. Products deleted from catalogue → category lost (extra ~4,730 rows)
  STRATEGY: DO NOT impute with 'Unknown'.
            Instead, try to recover from the child-to-parent relationship:
            for each configurable parent row, check if its child rows have a category.
            If child has category, copy to parent. Otherwise keep null.
  NOTE:     Full category recovery should be done via JOIN with product_catalogue
            table using product_id in the Feature Engineering step.

product_category / product_sub_category (64.8% null each):
  WHY NULL: Same dual cause as product_main_category.
  STRATEGY: Same — DO NOT impute with placeholder. Keep null.
""")

# ── Category recovery: configurable parent rows → inherit from their child rows
print("  Attempting category recovery: configurable parents → inherit from child rows...")

# Build a mapping: parent item_id → child's category (if child has it)
child_rows = df[df["parent_item_id"].notna()].copy()
category_map = (child_rows.dropna(subset=["product_main_category"])
                .groupby("parent_item_id")["product_main_category"]
                .first())
cat_map     = (child_rows.dropna(subset=["product_category"])
                .groupby("parent_item_id")["product_category"]
                .first())
subcat_map  = (child_rows.dropna(subset=["product_sub_category"])
                .groupby("parent_item_id")["product_sub_category"]
                .first())

# Apply: only fill configurable parent rows (row_type = configurable_parent)
parent_mask = (df["row_type"] == "configurable_parent") & df["product_main_category"].isnull()
recovered_main   = df.loc[parent_mask, "item_id"].map(category_map)
recovered_cat    = df.loc[parent_mask, "item_id"].map(cat_map)
recovered_subcat = df.loc[parent_mask, "item_id"].map(subcat_map)

df.loc[parent_mask, "product_main_category"] = recovered_main
df.loc[parent_mask, "product_category"]       = recovered_cat
df.loc[parent_mask, "product_sub_category"]   = recovered_subcat

after_main  = df["product_main_category"].isnull().sum()
after_cat   = df["product_category"].isnull().sum()
after_sub   = df["product_sub_category"].isnull().sum()

print(f"    product_main_category nulls: 21,811 → {after_main:,} "
      f"(recovered {21811 - after_main:,} configurable parent rows)")
print(f"    product_category nulls     : 23,137 → {after_cat:,} "
      f"(recovered {23137 - after_cat:,})")
print(f"    product_sub_category nulls : 23,137 → {after_sub:,} "
      f"(recovered {23137 - after_sub:,})")
print(f"    Remaining nulls = deleted-from-catalogue products (irrecoverable without external data)")
print(f"    Full recovery: JOIN with product_catalogue on product_id in Feature Engineering step")


# ════════════════════════════════════════════════════════════════════════════
# STEP 6 — DATA CLEANING
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 6 — Data Cleaning")
print("=" * 65)

# ── 6.1 NEGATIVE line_total_after_discount ────────────────────────────────
"""
EDA found 8 rows with negative line_total_after_discount.
Sample: 'Paracord Boot Laces' price=$8.00, discount=$7.60 → line_total=-$7.20
Root cause: discount_amount EXCEEDS the row_total (over-discounting).
These are mostly canceled orders (6 of 8 are canceled).
Strategy: Flag them with is_overdiscounted_line = 1.
          DO NOT set line_total to 0 — the negative value is real data.
          The downstream model must decide how to treat these (typically exclude
          from revenue sums but keep for cancellation analysis).
"""
df["is_overdiscounted_line"] = (df["line_total_after_discount"] < 0).astype(int)
n_neg = df["is_overdiscounted_line"].sum()
print(f"\n  Negative line_total_after_discount: {n_neg} rows flagged as 'is_overdiscounted_line'")
print(f"  Order status breakdown:")
print("  " + str(df.loc[df["is_overdiscounted_line"]==1, "order_status"].value_counts().to_dict()))
print(f"  NOT corrected — actual over-discounted lines (real data, mostly canceled)")

# ── 6.2 FLAG ZERO-PRICE REVENUE LINES ─────────────────────────────────────
"""
17,081 rows have price=0 AND row_total=0 — these are ALL configurable_parent rows.
row_type flag from Step 4 already captures this.
Add an explicit boolean for fast filtering in the modelling step.
"""
df["is_zero_price_line"] = (df["price"] == 0).astype(int)
print(f"\n  Zero-price lines: {df['is_zero_price_line'].sum():,}")
print(f"  row_type for zero-price lines:")
print("  " + str(df.loc[df["is_zero_price_line"]==1, "row_type"].value_counts().to_dict()))
print(f"  All zero-price lines are configurable_parent rows (confirmed by EDA)")

# ── 6.3 FLAG HIGH-VALUE LINES ─────────────────────────────────────────────
"""
EDA: price p99 (non-zero) = $360.00, row_total p99 = $379.81
181 rows above price p99 — genuine premium product lines (hiking packs, tents, GPS).
Strategy: Flag at p99, do NOT cap or remove.
"""
p99_price = df.loc[df["price"] > 0, "price"].quantile(0.99)
p99_row   = df.loc[df["row_total"] > 0, "row_total"].quantile(0.99)
df["is_high_value_line"] = ((df["price"] > p99_price) | (df["row_total"] > p99_row)).astype(int)
print(f"\n  High-value lines (above p99 price=${p99_price:.0f} or row_total=${p99_row:.0f}):")
print(f"  Flagged: {df['is_high_value_line'].sum():,} rows — NOT removed or capped")

# ── 6.4 FLAG BULK-ORDER LINES ─────────────────────────────────────────────
"""
EDA: qty_ordered max = 600, median = 1.
Lines with qty > 10 are bulk/B2B orders — very different demand profile.
Strategy: Flag them. Modelling step decides whether to separate B2C vs B2B.
"""
BULK_QTY_THRESHOLD = 10
df["is_bulk_line"] = (df["qty_ordered"] > BULK_QTY_THRESHOLD).astype(int)
print(f"\n  Bulk lines (qty_ordered > {BULK_QTY_THRESHOLD}): {df['is_bulk_line'].sum():,}")
print(f"  qty_ordered distribution for bulk lines:")
print("  " + str(df.loc[df["is_bulk_line"]==1, "qty_ordered"].describe().round(1).to_dict()))

# ── 6.5 FLAG CANCELED AND REFUNDED LINES ─────────────────────────────────
"""
EDA: 1,938 canceled lines (5.4%), 2,348 lines with qty_refunded > 0 (6.6%).
These are important labels for demand and churn modelling.
Flag them explicitly for easy downstream use.
"""
df["is_canceled_line"] = (df["order_status"] == "canceled").astype(int)
df["has_refund"]       = (df["qty_refunded"] > 0).astype(int)
df["has_cancellation"] = (df["qty_canceled"] > 0).astype(int)
print(f"\n  is_canceled_line : {df['is_canceled_line'].sum():,} rows ({df['is_canceled_line'].mean()*100:.1f}%)")
print(f"  has_refund       : {df['has_refund'].sum():,} rows ({df['has_refund'].mean()*100:.1f}%)")
print(f"  has_cancellation : {df['has_cancellation'].sum():,} rows ({df['has_cancellation'].mean()*100:.1f}%)")


# ════════════════════════════════════════════════════════════════════════════
# STEP 7 — STANDARDISE CATEGORICAL COLUMNS
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7 — Standardise Categorical Columns")
print("=" * 65)

# ── 7.1 CLEAN PAYMENT METHOD ─────────────────────────────────────────────
PAYMENT_MAP = {
    "braintree"                              : "braintree_credit",
    "braintree_paypal"                       : "paypal",
    "payment_services_paypal_smart_buttons"  : "paypal_smart",
    "payment_services_paypal_apple_pay"      : "apple_pay",
    "payment_services_paypal_google_pay"     : "google_pay",
    "free"                                   : "free",
    "checkmo"                                : "check_money_order",
    "cashondelivery"                         : "cash_on_delivery",
}
df["payment_method_clean"] = df["payment_method"].map(PAYMENT_MAP).fillna("other")
print(f"\n  payment_method_clean:")
print("  " + str(df["payment_method_clean"].value_counts().to_dict()))

# ── 7.2 CLEAN ORDER STATUS ────────────────────────────────────────────────
"""
order_status and order_state are near-identical (confirmed by EDA).
Both are kept but mapped to a cleaner 4-class lifecycle label.
"""
STATUS_MAP = {
    "complete"   : "completed",
    "canceled"   : "canceled",
    "closed"     : "returned",
    "pending"    : "pending",
    "processing" : "pending",
    "rejected"   : "canceled",
}
df["order_lifecycle"] = df["order_status"].map(STATUS_MAP).fillna("pending")
print(f"\n  order_lifecycle:")
print("  " + str(df["order_lifecycle"].value_counts().to_dict()))

# ── 7.3 LOWERCASE + STRIP ALL STRING COLS ─────────────────────────────────
str_cols = df.select_dtypes(include="object").columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip().str.lower()
print(f"\n  String standardisation (strip + lowercase) applied to {len(str_cols)} columns")


# ════════════════════════════════════════════════════════════════════════════
# STEP 8 — SORT & RESET INDEX
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8 — Sort and Reset Index")
print("=" * 65)

df.sort_values(["order_date", "order_id", "item_id"], ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"  Sorted by order_date → order_id → item_id")
print(f"  Date range: {df['order_date'].min().date()} → {df['order_date'].max().date()}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 9 — FINAL AUDIT
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 9 — Final Audit")
print("=" * 65)

print(f"\n  Final shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory       : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\n  Final columns ({df.shape[1]}):")
for i, col in enumerate(df.columns):
    dtype   = str(df[col].dtype)
    n_null  = df[col].isnull().sum()
    n_uniq  = df[col].nunique()
    print(f"  {i+1:2d}. {col:<35} dtype={dtype:<22} nulls={n_null:>6}  unique={n_uniq:>5}")

print(f"\n  Null summary (remaining — all justified by EDA):")
final_nulls = df.isnull().sum()
final_nulls = final_nulls[final_nulls > 0]
for col, cnt in final_nulls.items():
    pct = cnt / len(df) * 100
    print(f"    {col:<35}: {cnt:>6,}  ({pct:.1f}%)")

print(f"\n  New flag columns added (7):")
print(f"    row_type              : configurable_parent / revenue_line / simple_standalone")
print(f"    is_overdiscounted_line: discount > price (8 rows, mostly canceled)")
print(f"    is_zero_price_line    : price=0 configurable parent rows (17,081)")
print(f"    is_high_value_line    : price > p99=${p99_price:.0f} or row_total > p99=${p99_row:.0f}")
print(f"    is_bulk_line          : qty_ordered > {BULK_QTY_THRESHOLD} (B2B signal)")
print(f"    is_canceled_line      : order_status = 'canceled'")
print(f"    has_refund            : qty_refunded > 0")
print(f"    has_cancellation      : qty_canceled > 0")
print(f"    payment_method_clean  : short readable label for payment method")
print(f"    order_lifecycle       : 4-class order lifecycle label")

# Revenue sanity check on clean revenue lines only
rev_lines = df[df["row_type"].isin(["revenue_line", "simple_standalone"])]
print(f"\n  Revenue sanity check (revenue lines only, row_type != configurable_parent):")
print(f"    Revenue lines count   : {len(rev_lines):,} of {len(df):,} total lines")
print(f"    Total revenue         : ${rev_lines['row_total'].sum():>12,.2f}")
print(f"    Total line_total_disc : ${rev_lines['line_total_after_discount'].sum():>12,.2f}")
print(f"    Zero row_total        : {(rev_lines['row_total']==0).sum()}")
print(f"    Negative line_total   : {(rev_lines['line_total_after_discount']<0).sum()}")
print(f"    Avg price (non-zero)  : ${rev_lines.loc[rev_lines['price']>0,'price'].mean():>8,.2f}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 10 — SAVE
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 10 — Save Preprocessed Data")
print("=" * 65)

out_path = "order_items_preprocessed.csv"
df.to_csv(out_path, index=False)
print(f"  Saved → {out_path}")
print(f"  Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")


# ════════════════════════════════════════════════════════════════════════════
# PREPROCESSING SUMMARY
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("PREPROCESSING SUMMARY")
print("=" * 65)
print(f"""
  ORIGINAL  : 35,701 rows × 42 columns
  FINAL     : {df.shape[0]:,} rows × {df.shape[1]} columns
  ROWS REMOVED: 0  (no rows dropped — all are valid business records)

  DROPPED COLUMNS (8):
    Constant      (3): order_currency, order_currency_code, base_currency_code
    ETL metadata  (3): _extract_date, _run_id, _gold_loaded_at
    100% null     (1): product_attributes_json
    Redundant     (1): order_increment_id (same as order_id)

  NULL DECISIONS — 0 BLANKET FILLS:
    customer_id (74.1%)            → KEPT NULL (guest orders — no account)
    parent_item_id (52.2%)         → KEPT NULL (null = parent or standalone product)
    price_incl_tax (47.8%)         → KEPT NULL (null = configurable parent — no price)
    catalog_product_name (47.8%)   → KEPT NULL (use item_name instead — historical name)
    catalog_product_type (47.8%)   → KEPT NULL (use product_type — always populated)
    product_page_url (47.8%)       → KEPT NULL (display field, not ML feature)
    product_image_url (47.8%)      → KEPT NULL (display field, not ML feature)
    product_main_category (61.1%)  → PARTIAL RECOVERY via child→parent inheritance
                                     Full recovery: JOIN with product_catalogue in FE step
    product_category (64.8%)       → PARTIAL RECOVERY (same approach)
    product_sub_category (64.8%)   → PARTIAL RECOVERY (same approach)

  KEY ARCHITECTURAL FIX:
    row_type flag added to handle dual-row configurable architecture:
      'configurable_parent' : 17,081 rows — price=0, carries product context
      'revenue_line'        : 16,212 rows — actual price/revenue (child rows)
      'simple_standalone'   : 1,539 rows  — standalone products (maps, books etc.)
    RULE: Always filter df[row_type != 'configurable_parent'] for revenue analysis

  CLEANING DONE:
    Negative line_total_after_discount → flagged (8 over-discounted rows)
    Zero-price lines                   → flagged (17,081 configurable parents)
    High-value lines                   → flagged at p99 (not capped)
    Bulk-order lines (qty>10)          → flagged (B2B signal)
    Canceled lines                     → flagged
    Refund lines                       → flagged

  NEW COLUMNS ADDED (10):
    row_type, is_overdiscounted_line, is_zero_price_line, is_high_value_line,
    is_bulk_line, is_canceled_line, has_refund, has_cancellation,
    payment_method_clean, order_lifecycle

  NOT DONE (saved for FEATURE ENGINEERING step):
    → No revenue column (line_total_after_discount is already clean revenue)
    → No conversion rate features (view→cart etc.)
    → No aggregation to product or order level
    → No encoding (LabelEncoder / OneHotEncoder)
    → No scaling
    → No JOIN with product_catalogue for category recovery (use product_id join)
    → No log1p transform of price/row_total

  READY FOR: Feature Engineering → Model Training

  USE THIS IN NEXT STEP:
    items_clean = pd.read_csv('order_items_preprocessed.csv',
                              parse_dates=['order_date','created_at_ts','updated_at_ts'])
    revenue_lines = items_clean[items_clean['row_type'] != 'configurable_parent']
""")
print("✅ ORDER_ITEMS PREPROCESSING COMPLETE")

STEP 1 — Load Raw Data
  Raw shape  : 35,701 rows × 42 columns
  Memory     : 46.8 MB
  Each row   = one product line in an order (NOT one full order)
  Unique orders  : 14,176
  Unique products: 14,053

STEP 2 — Drop Zero-Value Columns

  Dropping 8 columns:
  Constant      (3): ['order_currency', 'order_currency_code', 'base_currency_code']
  ETL metadata  (3): ['_extract_date', '_run_id', '_gold_loaded_at']
  100% null     (1): ['product_attributes_json']
  Redundant     (1): ['order_increment_id']

  Shape after drop : 35,701 rows × 34 columns

STEP 3 — Fix Data Types
  created_at_ts  → datetime64  ✓
  updated_at_ts  → datetime64  ✓
  order_date     → already datetime64  ✓
  customer_is_guest → already bool  ✓
  parent_item_id → float64 with nulls (null = standalone product)  ✓
  price_incl_tax → float64 with nulls (null = configurable parent row)  ✓

  Dtypes after fix:
item_id                               int64
order_id                              int64
order_date              

In [ ]:
"""
============================================================
CUSTOMERS TABLE — PREPROCESSING ONLY (NO FEATURE ENGINEERING)
apeak.ai | Outdoor Retail | AWS Gold Tables
27,721 registered customers | Feb 2014 – Apr 2026
============================================================
WHAT THIS SCRIPT DOES:
  → Drops columns with zero analytical value (constants, ETL metadata,
    PII tokens, 100%-null fields, near-constant fields)
  → Fixes data types to their correct forms
  → Handles nulls with the correct strategy per column
    — justified by EDA, no blanket fill
  → Flags dev/test accounts (virtually the entire table is dev data)
  → Cleans and standardises categorical values

WHAT THIS SCRIPT DOES NOT DO:
  → No feature engineering (no tenure, no order count, no CLTV)
  → No encoding (LabelEncoder / OneHotEncoder)
  → No scaling
  → No aggregation or JOIN with orders table
  → No RFM derivation

KEY EDA FINDING THIS PREPROCESSING IS BUILT AROUND:
  This table contains 27,721 registered customer accounts.
  However, 99.98% of all rows use development/test email domains:
    krishtechnolabs.net  : 26,705 rows  (96.3%)
    example.com          :  1,000 rows   (3.6%)  — batch-created 2026-03-05
    krishtechnolabs.com  :     10 rows
    krish.com            :      1 row
    gmail.com / yopmail  :      5 rows   ← only 5 real customer accounts
  This means the customers table DOES NOT contain real end-customers.
  It is entirely composed of dev team accounts and synthetic test data.
  Strategy: flag all rows as is_dev_account, preserve the table structure,
  and expose this finding clearly so the modelling step can decide how to
  handle (likely: use orders.customer_id as the CLTV entity, not this table).
============================================================
"""
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)

# ════════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD DATA
# ════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 — Load Raw Data")
print("=" * 65)

FILE   = "AWS Gold Tables (1).xlsx"
sheets = pd.read_excel(FILE, sheet_name=None, engine="openpyxl")
df     = sheets["customers"].copy()

print(f"  Raw shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory     : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Each row   = one registered Magento customer account")
print(f"  Note       : Guest orders are NOT in this table — they appear in")
print(f"               the orders table with customer_id = NULL")

# ════════════════════════════════════════════════════════════════════════════
# STEP 2 — DROP ZERO-VALUE COLUMNS
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 2 — Drop Zero-Value Columns")
print("=" * 65)

"""
WHY WE DROP THESE COLUMNS — EACH DECISION JUSTIFIED BY EDA:

CATEGORY A — CONSTANT COLUMNS (single unique value or near-zero variance):
  customer_group_id    : always 1 (single group — 'General') — 27,721/27,721
  gender               : 99.98% null AND all 5 non-null values are 0.0
                         (0.0 = 'Not Specified' in Magento — not actually Male=1)
                         → zero variance in non-null values AND near-total null
  disable_auto_group_change : 27,720 False / 1 True — zero variance for any model

CATEGORY B — ETL PIPELINE METADATA (not business data):
  _extract_date   : ETL batch date — 3 unique values but they are run timestamps
  _run_id         : ETL run identifier — 3 unique values (same 3 runs)
  _gold_loaded_at : single value '2026-04-21 05:15:30.647 UTC' for all rows
  → Operational fields, zero analytical signal

CATEGORY C — PII TOKENS (hashed, not usable for analysis):
  email_token                 : SHA-256 hash of email — 27,721 unique, not groupable
  firstname_token             : SHA-256 hash of first name — not usable
  lastname_token              : SHA-256 hash of last name — not usable
  → These tokens protect PII but are not decodable.
    No two tokens can be joined unless you hash the same plaintext.
    Unique per row — zero grouping or signal value for modelling.
    The email_domain column (retained) gives us all the signal we can extract.

CATEGORY D — 99.97% NULL ADDRESS FIELDS (only 7 rows have data):
  default_billing_address_id      : 27,714 null (99.97%) — 7 dev-team addresses
  default_billing_country_id      : 27,714 null (99.97%) — all 7 non-null = 'US'
  default_billing_region          : 27,714 null (99.97%) — JSON-encoded dicts
  default_billing_city            : 27,714 null (99.97%) — 3 cities: ahmedabad, california, Little Rock
  default_billing_postcode_token  : 27,714 null (99.97%) — hashed, not usable
  default_billing_telephone_token : 27,714 null (99.97%) — hashed, not usable
  default_shipping_address_id     : 27,714 null (99.97%) — same 7 rows as billing
  default_shipping_country_id     : 27,714 null (99.97%) — all 'US'
  default_shipping_region         : 27,714 null (99.97%) — JSON-encoded dicts
  default_shipping_city           : 27,714 null (99.97%) — same 3 dev cities
  default_shipping_postcode_token : 27,714 null (99.97%) — hashed
  default_shipping_telephone_token: 27,714 null (99.97%) — hashed
  addresses_masked_json           : 27,714 null (99.97%) — JSON blob of above
  → Cannot impute 99.97% missing with any meaningful strategy.
    The 7 non-null rows are all dev-team accounts (ahmedabad = dev city).
    No real customer ever completed their address profile.

CATEGORY E — NEAR-CONSTANT AFTER EDA:
  store_id    : 27,718 = 1 (single store), 3 = 0 (Admin-created dev rows)
  website_id  : 27,720 = 1, 1 = 0 (same Admin dev rows)
  created_in  : 27,718 = 'Default Store View', 3 = 'Admin'
                → The 3 Admin rows are ALL dev accounts (confirmed: email_domain = krishtechnolabs.net)
                → Zero business variance; all customers are from the same store view
"""

COLS_CONSTANT = [
    "customer_group_id",           # always 1 ('General' group)
    "gender",                      # 99.98% null; 5 non-nulls all = 0.0 (unspecified)
    "disable_auto_group_change",   # 27,720 False / 1 True — zero variance
]
COLS_ETL_META = [
    "_extract_date",               # ETL batch date — operational metadata
    "_run_id",                     # ETL run ID — operational metadata
    "_gold_loaded_at",             # single value — load timestamp
]
COLS_PII_TOKENS = [
    "email_token",                 # SHA-256 hash of email — not decodable
    "firstname_token",             # SHA-256 hash of first name — not decodable
    "lastname_token",              # SHA-256 hash of last name — not decodable
]
COLS_HIGH_NULL_ADDRESS = [
    "default_billing_address_id",
    "default_billing_country_id",
    "default_billing_region",
    "default_billing_city",
    "default_billing_postcode_token",
    "default_billing_telephone_token",
    "default_shipping_address_id",
    "default_shipping_country_id",
    "default_shipping_region",
    "default_shipping_city",
    "default_shipping_postcode_token",
    "default_shipping_telephone_token",
    "addresses_masked_json",       # JSON blob of the above — also 99.97% null
]
COLS_NEAR_CONSTANT = [
    "store_id",                    # 99.99% = 1 (single store); 3 Admin-dev rows = 0
    "website_id",                  # 99.99% = 1; 1 Admin row = 0
    "created_in",                  # 'Default Store View' (99.99%); 3 Admin dev rows
]

ALL_DROP = (COLS_CONSTANT + COLS_ETL_META + COLS_PII_TOKENS
            + COLS_HIGH_NULL_ADDRESS + COLS_NEAR_CONSTANT)

print(f"\n  Dropping {len(ALL_DROP)} columns:")
print(f"  Constant         ({len(COLS_CONSTANT)}): {COLS_CONSTANT}")
print(f"  ETL metadata     ({len(COLS_ETL_META)}): {COLS_ETL_META}")
print(f"  PII tokens       ({len(COLS_PII_TOKENS)}): {COLS_PII_TOKENS}")
print(f"  High-null addr   ({len(COLS_HIGH_NULL_ADDRESS)}): {COLS_HIGH_NULL_ADDRESS}")
print(f"  Near-constant    ({len(COLS_NEAR_CONSTANT)}): {COLS_NEAR_CONSTANT}")

df.drop(columns=ALL_DROP, inplace=True)

print(f"\n  Shape after drop : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Remaining columns: {df.columns.tolist()}")

# ════════════════════════════════════════════════════════════════════════════
# STEP 3 — FIX DATA TYPES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3 — Fix Data Types")
print("=" * 65)

"""
TYPE DECISIONS:
  customer_id           → int64 — already correct, canonical join key
  customer_group_id     → DROPPED (constant)
  is_subscribed         → already bool — CORRECT (no nulls in this table;
                          unlike orders where guest = null, here every
                          registered customer has a known subscription status)
  created_at_ts         → string '2024-06-08 00:25:20.000 UTC' → parse to datetime
  updated_at_ts         → same → parse to datetime
  customer_created_date → already datetime64 from read_excel — CORRECT
  customer_updated_date → already datetime64 from read_excel — CORRECT
  email_domain          → str — keep as-is for categorical analysis
"""

df["created_at_ts"] = pd.to_datetime(df["created_at_ts"], errors="coerce", utc=True)
df["updated_at_ts"] = pd.to_datetime(df["updated_at_ts"], errors="coerce", utc=True)

# Remove timezone info for consistency with orders/order_items tables (all UTC)
df["created_at_ts"] = df["created_at_ts"].dt.tz_localize(None)
df["updated_at_ts"] = df["updated_at_ts"].dt.tz_localize(None)

print("  created_at_ts       → datetime64  ✓")
print("  updated_at_ts       → datetime64  ✓")
print("  customer_created_date → already datetime64  ✓")
print("  customer_updated_date → already datetime64  ✓")
print("  is_subscribed         → already bool (no nulls — all registered customers have status)  ✓")
print("  customer_id           → int64 (canonical join key)  ✓")
print("  email_domain          → str (categorical)  ✓")
print(f"\n  Dtypes after fix:")
print(df.dtypes.to_string())

# ════════════════════════════════════════════════════════════════════════════
# STEP 4 — NULL HANDLING
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4 — Null Handling (Justified per Column)")
print("=" * 65)

print("""
NULL STRATEGY DECISIONS (from EDA):

No remaining columns have meaningful null values after the column drop in Step 2.

  customer_id           : 0 nulls — every registered customer has an ID
  customer_group_id     : DROPPED (constant = 1)
  email_domain          : 0 nulls — every account has a domain
  is_subscribed         : 0 nulls — every registered account has a known status
  created_at_ts         : 0 nulls
  updated_at_ts         : 0 nulls
  customer_created_date : 0 nulls
  customer_updated_date : 0 nulls

All address fields (99.97% null) were DROPPED in Step 2 — cannot impute.

IMPORTANT NOTE:
  Virtually all 'null' information in this dataset reflects the fact that
  99.98% of accounts are dev/test accounts that never completed real profiles.
  The 7 accounts with address data are all krishtechnolabs dev team members.
  Address field recovery via JOIN with other tables is not viable here.
""")

nulls = df.isnull().sum()
if nulls.sum() == 0:
    print("  ✓ No nulls in remaining columns — nothing to handle")
else:
    print("  Null counts:")
    print(nulls[nulls > 0].to_string())

# ════════════════════════════════════════════════════════════════════════════
# STEP 5 — FLAG DEV / TEST ACCOUNTS
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 5 — Flag Dev / Test Accounts")
print("=" * 65)

"""
EDA FINDING — CRITICAL:
  This table is composed almost entirely of development team and synthetic
  test accounts. Real customer email domains account for only 5 rows (0.02%):

  Domain breakdown:
    krishtechnolabs.net  : 26,705  (96.3%)  ← dev team
    example.com          :  1,000   (3.6%)  ← 1,000 accounts batch-created on
                                              2026-03-05 (single day) — synthetic
                                              test data load, NOT real customers
    krishtechnolabs.com  :     10           ← dev team
    krish.com            :      1           ← dev team
    gmail.com            :      3           ← potentially real (cannot confirm)
    yopmail.com          :      2           ← yopmail = disposable email service,
                                              likely test accounts

  STRATEGY:
    Flag all accounts by their account type so the modelling step can choose
    its filter policy. Three tiers:
      'dev_team'    → krishtechnolabs.net / .com / krish.com (known dev domains)
      'synthetic'   → example.com accounts (1,000 batch-created on one day)
      'real'        → gmail.com / yopmail.com (5 accounts, likely real or test)

  WHY NOT REMOVE?
    We never remove rows in preprocessing. The modelling step decides:
      → For CLTV: exclude all (df[df.account_type=='real']) — only 5 remain
      → For testing the pipeline: use df[df.account_type!='real']
      → For data auditing: inspect all tiers
"""

def classify_account(domain):
    dev_domains_exact = {"krishtechnolabs.net", "krishtechnolabs.com", "krish.com"}
    if domain in dev_domains_exact:
        return "dev_team"
    elif domain == "example.com":
        return "synthetic"
    else:
        return "real"   # gmail.com, yopmail.com

df["account_type"] = df["email_domain"].apply(classify_account)

print(f"\n  account_type distribution:")
for at, cnt in df["account_type"].value_counts().items():
    print(f"    {at:<12}: {cnt:>6,}  ({cnt/len(df)*100:.1f}%)")

print(f"""
  ⚠  WARNING FOR MODELLING STEP:
     Only {(df['account_type']=='real').sum()} rows are classified as 'real' customers.
     The customers table CANNOT be used as the primary customer entity for CLTV.
     Use orders.customer_id (registered orders) as the CLTV entity instead.
     The customers table provides supplementary attributes (is_subscribed,
     created_at_ts tenure anchor) for those registered customer IDs.
""")

# Flag accounts created in a synthetic batch (1,000 example.com on same day)
df["is_synthetic_batch"] = (
    (df["email_domain"] == "example.com") &
    (df["customer_created_date"] == pd.Timestamp("2026-03-05"))
).astype(int)
print(f"  is_synthetic_batch = 1: {df['is_synthetic_batch'].sum():,} "
      f"(example.com accounts all created on 2026-03-05)")

# ════════════════════════════════════════════════════════════════════════════
# STEP 6 — DATA CLEANING
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 6 — Data Cleaning")
print("=" * 65)

# ── 6.1 VALIDATE customer_id UNIQUENESS ──────────────────────────────────
"""
customer_id is the primary key for this table and the join key to orders.
Verify there are no duplicates before any downstream join.
"""
n_dupes = df["customer_id"].duplicated().sum()
print(f"\n  customer_id duplicates: {n_dupes}  (should be 0)")
if n_dupes > 0:
    print("  ⚠  DUPLICATE customer_ids FOUND — investigate before joining to orders")
    print(df[df["customer_id"].duplicated(keep=False)][["customer_id"]].to_string())
else:
    print("  ✓ customer_id is unique — safe to use as JOIN key with orders table")

# ── 6.2 VALIDATE TIMESTAMP CONSISTENCY ───────────────────────────────────
"""
customer_created_date should equal the date part of created_at_ts.
Any mismatch indicates a data pipeline issue (e.g., timezone shift at midnight).
"""
ts_date = df["created_at_ts"].dt.normalize()
ccd     = pd.to_datetime(df["customer_created_date"])
mismatch_dates = (ts_date != ccd).sum()
print(f"\n  Timestamp vs date mismatch (created_at_ts.date != customer_created_date): "
      f"{mismatch_dates:,} rows")
if mismatch_dates > 0:
    print(f"  Note: Minor off-by-one mismatches expected due to UTC vs local timezone "
          f"boundary — not a data error, just timezone normalization artifact")
print(f"  Recommendation: Use created_at_ts (full precision) for tenure calculations "
      f"in Feature Engineering step")

# ── 6.3 VALIDATE updated_at ORDERING ─────────────────────────────────────
"""
updated_at_ts must be >= created_at_ts for every row.
Violations would indicate ETL pipeline clock drift or data corruption.
"""
update_before_create = (df["updated_at_ts"] < df["created_at_ts"]).sum()
print(f"\n  updated_at_ts < created_at_ts violations: {update_before_create:,} rows")

# ── 6.4 FLAG ADMIN-CREATED ACCOUNTS ──────────────────────────────────────
"""
3 accounts have store_id=0 and were created via the 'Admin' panel
rather than the storefront. These are test/system accounts.
'created_in' column was dropped (near-constant) but we preserve the signal
in a new flag: is_admin_created.
"""
# We need to re-derive this from created_at timing + known admin customer_ids
# Since created_in column was dropped, we identify them from store_id=0
# (already dropped) — reconstruct from raw knowledge via EDA
admin_customer_ids = {16388, 19294, 1467}  # identified in EDA (store_id=0, created_in='Admin')
df["is_admin_created"] = df["customer_id"].isin(admin_customer_ids).astype(int)
print(f"\n  Admin-created accounts flagged: {df['is_admin_created'].sum()} "
      f"(store_id=0, created via Magento Admin panel)")

# ════════════════════════════════════════════════════════════════════════════
# STEP 7 — STANDARDISE CATEGORICAL COLUMNS
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7 — Standardise Categorical Columns")
print("=" * 65)

# ── 7.1 LOWERCASE + STRIP ALL STRING COLS ─────────────────────────────────
"""
Standardise all object columns: strip whitespace, lowercase.
Prevents mismatches like 'Gmail.com' vs 'gmail.com' in groupby/join.
"""
str_cols = df.select_dtypes(include="object").columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip().str.lower()

print(f"\n  String standardisation (strip + lowercase) applied to {len(str_cols)} columns:")
print(f"  {str_cols}")

# Verify email_domain post-clean
print(f"\n  email_domain values after standardisation:")
print("  " + str(df["email_domain"].value_counts().to_dict()))

# account_type values after lowercase
print(f"\n  account_type values after standardisation:")
print("  " + str(df["account_type"].value_counts().to_dict()))

# ════════════════════════════════════════════════════════════════════════════
# STEP 8 — SORT & RESET INDEX
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8 — Sort and Reset Index")
print("=" * 65)

df.sort_values("customer_created_date", ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"  Sorted by customer_created_date ascending (earliest first)")
print(f"  Index reset to 0-based continuous integer")
print(f"  Date range: {df['customer_created_date'].min().date()} → "
      f"{df['customer_created_date'].max().date()}")

# ════════════════════════════════════════════════════════════════════════════
# STEP 9 — FINAL AUDIT
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 9 — Final Audit")
print("=" * 65)

print(f"\n  Final shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory       : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\n  Final columns ({df.shape[1]}):")
for i, col in enumerate(df.columns):
    dtype   = str(df[col].dtype)
    n_null  = df[col].isnull().sum()
    n_uniq  = df[col].nunique()
    print(f"  {i+1:2d}. {col:<35} dtype={dtype:<22} nulls={n_null:>6}  unique={n_uniq:>5}")

print(f"\n  Null summary:")
final_nulls = df.isnull().sum()
if final_nulls.sum() == 0:
    print("    ✓ No nulls in any retained column")
else:
    for col, cnt in final_nulls[final_nulls > 0].items():
        print(f"    {col:<35}: {cnt:>6,}  ({cnt/len(df)*100:.1f}%)")

print(f"\n  New columns added by preprocessing:")
print(f"    account_type       : 'dev_team' / 'synthetic' / 'real'")
print(f"    is_synthetic_batch : 1 = 1,000 example.com accounts created in bulk on 2026-03-05")
print(f"    is_admin_created   : 1 = created via Magento Admin panel (store_id=0)")

print(f"\n  Account type distribution:")
for at, cnt in df["account_type"].value_counts().items():
    print(f"    {at:<12}: {cnt:>6,}  ({cnt/len(df)*100:.2f}%)")

print(f"\n  Subscription status:")
print(f"    is_subscribed = True  : {df['is_subscribed'].sum():>6,}  "
      f"({df['is_subscribed'].mean()*100:.1f}%)")
print(f"    is_subscribed = False : {(~df['is_subscribed']).sum():>6,}  "
      f"({(~df['is_subscribed']).mean()*100:.1f}%)")
print(f"    NOTE: These are ALL dev/test accounts — subscription data not meaningful")

print(f"\n  Customer tenure range (using created_at_ts):")
print(f"    Earliest account : {df['created_at_ts'].min()}")
print(f"    Latest account   : {df['created_at_ts'].max()}")

# ════════════════════════════════════════════════════════════════════════════
# STEP 10 — SAVE
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 10 — Save Preprocessed Data")
print("=" * 65)

out_path = "customers_preprocessed.csv"
df.to_csv(out_path, index=False)
print(f"  Saved → {out_path}")
print(f"  Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")

# ════════════════════════════════════════════════════════════════════════════
# PREPROCESSING SUMMARY
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("PREPROCESSING SUMMARY")
print("=" * 65)
print(f"""
  ORIGINAL  : 27,721 rows × 32 columns
  FINAL     : {df.shape[0]:,} rows × {df.shape[1]} columns
  ROWS REMOVED: 0  (no rows dropped — all accounts are valid records)

  DROPPED COLUMNS (22):
    Constant         (3): customer_group_id (always 1), gender (99.98% null + all = 0.0),
                          disable_auto_group_change (27,720 False / 1 True)
    ETL metadata     (3): _extract_date, _run_id, _gold_loaded_at
    PII tokens       (3): email_token, firstname_token, lastname_token
                          (SHA-256 hashed — not decodable or groupable)
    High-null addr  (13): all billing/shipping address columns (99.97% null)
                          + addresses_masked_json (same sparsity)
    Near-constant    (3): store_id (always 1 except 3 Admin rows),
                          website_id (always 1 except 1 Admin row),
                          created_in (always 'Default Store View' except 3 Admin rows)

  NULL DECISIONS — 0 BLANKET FILLS:
    No remaining columns have nulls after the column drop.
    All address columns (99.97% null) were dropped — irrecoverable without
    a separate address enrichment source.

  KEY FINDING — DEV TABLE WARNING:
    99.98% of all customer accounts are dev team or synthetic test data.
    Only 5 rows use non-dev email domains (gmail.com / yopmail.com).
    The customers table SHOULD NOT be used as the primary CLTV entity.
    Use orders.customer_id (registered orders) as the customer entity instead.
    This table is useful for:
      → is_subscribed attribute (join to registered order customer_ids)
      → created_at_ts as tenure anchor (account age at time of first order)
      → is_admin_created flag (exclude system accounts)

  CLEANING DONE:
    customer_id uniqueness verified — ✓ no duplicates (safe join key)
    timestamp vs date consistency checked
    updated_at vs created_at ordering validated
    Admin-created accounts flagged (3 rows)

  NEW COLUMNS ADDED (3):
    account_type       : dev_team / synthetic / real
    is_synthetic_batch : 1 = bulk example.com test data (2026-03-05)
    is_admin_created   : 1 = created via Admin panel

  NOT DONE (saved for FEATURE ENGINEERING step):
    → No account tenure / customer age calculation
    → No JOIN with orders for order count, CLTV, first/last order date
    → No encoding (LabelEncoder / OneHotEncoder)
    → No scaling
    → No RFM derivation

  READY FOR: Feature Engineering → Model Training

  USE THIS IN NEXT STEP:
    customers_clean = pd.read_csv('customers_preprocessed.csv',
                                  parse_dates=['created_at_ts', 'updated_at_ts',
                                               'customer_created_date', 'customer_updated_date'])
    # For CLTV join — get subscription status and account type for registered customers:
    # orders_with_attrs = orders_clean.merge(
    #     customers_clean[['customer_id','is_subscribed','account_type','created_at_ts']],
    #     on='customer_id', how='left'
    # )
""")

print("✅ CUSTOMERS PREPROCESSING COMPLETE")

STEP 1 — Load Raw Data
  Raw shape  : 27,721 rows × 32 columns
  Memory     : 33.5 MB
  Each row   = one registered Magento customer account
  Note       : Guest orders are NOT in this table — they appear in
               the orders table with customer_id = NULL

STEP 2 — Drop Zero-Value Columns

  Dropping 25 columns:
  Constant         (3): ['customer_group_id', 'gender', 'disable_auto_group_change']
  ETL metadata     (3): ['_extract_date', '_run_id', '_gold_loaded_at']
  PII tokens       (3): ['email_token', 'firstname_token', 'lastname_token']
  High-null addr   (13): ['default_billing_address_id', 'default_billing_country_id', 'default_billing_region', 'default_billing_city', 'default_billing_postcode_token', 'default_billing_telephone_token', 'default_shipping_address_id', 'default_shipping_country_id', 'default_shipping_region', 'default_shipping_city', 'default_shipping_postcode_token', 'default_shipping_telephone_token', 'addresses_masked_json']
  Near-constant    (3): ['sto

In [ ]:
"""
============================================================
PRODUCT_CATALOGUE TABLE — PREPROCESSING ONLY (NO FEATURE ENGINEERING)
apeak.ai | Outdoor Retail | AWS Gold Tables
5,996 products | Aug 2020 – Apr 2026
============================================================
WHAT THIS SCRIPT DOES:
  → Drops columns with zero analytical value (constants, ETL metadata,
    100%-null fields, fully redundant columns)
  → Fixes data types to their correct forms
  → Handles nulls with the correct strategy per column
    — justified by EDA, no blanket fills
  → Flags structural anomalies without removing rows
  → Cleans and standardises categorical values

WHAT THIS SCRIPT DOES NOT DO:
  → No feature engineering (no discount_pct, no price bands, no demand score)
  → No encoding (LabelEncoder / OneHotEncoder)
  → No scaling
  → No aggregation or JOIN with order_items for sales velocity
  → No parsing of JSON blobs (configurable_options, variants) — saved for FE step

KEY EDA FINDINGS THIS PREPROCESSING IS BUILT AROUND:

  1. FLAT CATALOGUE ARCHITECTURE — NOT A HIERARCHY TABLE:
     parent_product_id, parent_id, parent_sku are ALL 100% null.
     Unlike order_items (where parent_item_id links child→parent rows),
     this catalogue table stores products flat — one row per sellable unit.
     Configurable products (4,499 rows) are standalone catalogue entries.
     Simple products (1,497 rows) are standalone entries too.
     The parent-child relationship exists only in the 'variants' JSON column.

  2. product_type IS NOT MAGENTO'S type_id:
     - type_id   : Magento's actual product architecture type ('configurable' / 'simple')
     - product_type : the Magento attribute_set label — a merchandise subcategory
       (e.g., 'U.S.G.S Topo Maps', 'Casual', 'Daypacks', 'Ballcaps')
     Both columns are retained. product_type ≈ attribute_set_name (277 unique values).

  3. special_price SEMANTICS:
     - 57.9% null = no promotional price set (normal pricing)
     - 35.6% of rows: special_price == price (no actual discount — Magento placeholder)
     - 6.5% of rows: special_price < price (genuine promotional discount)
     - 5 rows: special_price > price (data anomaly — flagged)
     Strategy: flag genuine discounts rather than dropping or imputing.

  4. 82 OUT-OF-STOCK CONFIGURABLE PRODUCTS:
     quantity=0 and is_in_stock=False are perfectly correlated (82 rows each).
     These products have no variants (variants=null), price=null — they are
     discontinued/delisted products still present in the catalogue.
     Strategy: flag as is_discontinued rather than removing.

  5. DUPLICATE product_names (286 rows, 143 products):
     EDA confirmed: most are the SAME product listed under different
     attribute_set configurations (different colour/size variant groupings),
     not genuine duplicate entries. product_id and sku are always unique.
     Strategy: flag as has_name_duplicate — resolve via product_id in FE step.

  6. short_description == description for 100% of rows:
     Both columns are identical. Drop short_description as pure redundancy.

  7. meta_title vs product_name:
     meta_title = '{brand_name} {product_name}' (prepends brand prefix).
     0 rows have meta_title == product_name — they always differ.
     meta_title is an SEO field, not a cleaner product name.
     Both retained; feature engineering step extracts brand from meta_title if needed.

  8. category_path_names vs categories:
     99.9% identical after bracket-stripping. categories is the cleaner form.
     Drop category_path_names as redundant.
============================================================
"""
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)

# ════════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD DATA
# ════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 — Load Raw Data")
print("=" * 65)

FILE   = "AWS Gold Tables (1).xlsx"
sheets = pd.read_excel(FILE, sheet_name=None, engine="openpyxl")
df     = sheets["product_catalogue"].copy()

print(f"  Raw shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory     : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Each row   = one product in the catalogue (configurable or simple)")
print(f"  Unique products : {df['product_id'].nunique():,} (by product_id)")
print(f"  type_id split   : {df['type_id'].value_counts().to_dict()}")

# ════════════════════════════════════════════════════════════════════════════
# STEP 2 — DROP ZERO-VALUE COLUMNS
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 2 — Drop Zero-Value Columns")
print("=" * 65)

"""
WHY WE DROP THESE COLUMNS — EACH DECISION JUSTIFIED BY EDA:

CATEGORY A — CONSTANT COLUMNS (single unique value, zero variance):
  status      : always 'Enabled' (5,996/5,996) — all products are active in catalogue
  visibility  : always 'Catalog, Search' (5,996/5,996) — all products are publicly visible
  weight      : always 0 (5,996/5,996) — Magento weight field never populated (digital/flat)
  main_category_root_id : always 1.0 (5,991 non-null; 5 null = uncategorised products)
                          root_id=1 is the Magento catalogue root — zero variance

CATEGORY B — ETL PIPELINE METADATA (not business data):
  _extract_date   : ETL batch date — 3 unique values (all 2026 pipeline runs)
  _run_id         : ETL run identifier — 3 unique values
  _gold_loaded_at : single timestamp '2026-04-21 05:15:36.793 UTC' for all rows
  → Operational pipeline fields, no analytical signal

CATEGORY C — 100% NULL (entire column is empty):
  parent_product_id       : 100% null — this is a FLAT catalogue, no hierarchy
  parent_id               : 100% null — same reason
  parent_sku              : 100% null — same reason
  url_path                : 100% null — Magento URL path field never populated
  main_category_is_active : 100% null — ETL enrichment not completed
  additional_attributes_json : 100% null — ETL enrichment not completed

CATEGORY D — FULLY REDUNDANT COLUMNS (duplicates existing information exactly):
  short_description : 100% identical to 'description' column (confirmed by EDA:
                      all 5,996 rows have short_description == description)
                      → Keeping both doubles storage with zero added signal
  category_path_names : 99.9% identical to 'categories' column after bracket removal
                        → categories is the cleaner comma-separated format; use that
  url_key           : already fully encoded in product_page_url as the path segment
                      (e.g. url_key='texas-paisley-bandana-715715' maps 1:1 to the URL)
                      AND url_key = product_page_url without domain + .html extension
                      → product_page_url is the complete, usable field; url_key is redundant
"""

COLS_CONSTANT = [
    "status",                     # always 'Enabled'
    "visibility",                 # always 'Catalog, Search'
    "weight",                     # always 0 (digital-only products, weight never set)
    "main_category_root_id",      # always 1.0 (Magento catalogue root ID)
]
COLS_ETL_META = [
    "_extract_date",              # ETL batch date
    "_run_id",                    # ETL run ID
    "_gold_loaded_at",            # single ETL load timestamp
]
COLS_100PCT_NULL = [
    "parent_product_id",          # 100% null — flat catalogue, no hierarchy
    "parent_id",                  # 100% null — same
    "parent_sku",                 # 100% null — same
    "url_path",                   # 100% null — Magento field never populated
    "main_category_is_active",    # 100% null — ETL enrichment not done
    "additional_attributes_json", # 100% null — ETL enrichment not done
]
COLS_REDUNDANT = [
    "short_description",    # 100% identical to 'description' — confirmed by EDA
    "category_path_names",  # 99.9% same as 'categories' (bracket-format variant)
    "url_key",              # redundant with product_page_url (url_key IS the URL slug)
]

ALL_DROP = COLS_CONSTANT + COLS_ETL_META + COLS_100PCT_NULL + COLS_REDUNDANT

print(f"\n  Dropping {len(ALL_DROP)} columns:")
print(f"  Constant       ({len(COLS_CONSTANT)}): {COLS_CONSTANT}")
print(f"  ETL metadata   ({len(COLS_ETL_META)}): {COLS_ETL_META}")
print(f"  100% null      ({len(COLS_100PCT_NULL)}): {COLS_100PCT_NULL}")
print(f"  Redundant      ({len(COLS_REDUNDANT)}): {COLS_REDUNDANT}")

df.drop(columns=ALL_DROP, inplace=True)

print(f"\n  Shape after drop : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Remaining columns: {df.columns.tolist()}")

# ════════════════════════════════════════════════════════════════════════════
# STEP 3 — FIX DATA TYPES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3 — Fix Data Types")
print("=" * 65)

"""
TYPE DECISIONS:
  product_id      → int64 — already correct, canonical join key to order_items
  sku             → object (str) — already correct; SKU is a mix of int and string
                    values (e.g. '715715', 'pg58307') — keep as string, not int
  type_id         → str (object) — already correct ('configurable' / 'simple')
  product_type    → str (object) — already correct (attribute_set label)
  attribute_set_id→ int64 — already correct
  price           → float64 — already correct (has 55 nulls for OOS products)
  special_price   → float64 — already correct (57.9% null = no promo price)
  quantity        → int64 — already correct
  is_in_stock     → bool — already correct
  updated_at_ts   → string '2025-05-17 19:23:14.000 UTC' → parse to datetime
  updated_date    → already datetime64 from read_excel — CORRECT

  IMPORTANT — product_type column:
    Despite being named 'product_type', this is NOT the Magento product type.
    It is the attribute_set label (merchandising subcategory).
    True Magento product type is in the 'type_id' column.
    This naming confusion is a known ETL schema issue — no rename here
    (renaming is a FE step decision), but we document it clearly.
"""

df["updated_at_ts"] = pd.to_datetime(df["updated_at_ts"], errors="coerce", utc=True)
df["updated_at_ts"] = df["updated_at_ts"].dt.tz_localize(None)

print("  updated_at_ts    → datetime64  ✓")
print("  updated_date     → already datetime64  ✓")
print("  product_id       → int64 (canonical join key to order_items)  ✓")
print("  sku              → str (mixed int+str values, keep as string)  ✓")
print("  type_id          → str ('configurable'/'simple')  ✓")
print("  product_type     → str (attribute_set label, NOT Magento type_id)  ✓")
print("  price            → float64 (55 nulls = OOS discontinued products)  ✓")
print("  special_price    → float64 (57.9% null = no promo set)  ✓")
print("  is_in_stock      → bool  ✓")
print(f"\n  Dtypes after fix:")
print(df.dtypes.to_string())

# ════════════════════════════════════════════════════════════════════════════
# STEP 4 — NULL HANDLING (JUSTIFIED PER COLUMN, NO BLANKET FILLS)
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4 — Null Handling (Justified per Column)")
print("=" * 65)

print("""
NULL STRATEGY DECISIONS — EVERY DECISION JUSTIFIED BY EDA:

price (55 nulls = 0.9%):
  WHY NULL: EDA confirmed — ALL 55 null-price rows are configurable products
            with quantity=0 and is_in_stock=False. These are discontinued or
            delisted products where Magento clears the price on deactivation.
  STRATEGY: DO NOT impute. Null price + OOS = discontinued product signal.
            The is_discontinued flag (added in Step 5) encodes this.
  REASON:   Imputing mean/median price would fabricate revenue estimates for
            products that are not actually for sale.

special_price (3,471 nulls = 57.9%):
  WHY NULL: No promotional pricing has been set for these products.
            Null special_price = 'this product is at its regular price'.
            This is intentional Magento behaviour — not a data quality issue.
  STRATEGY: DO NOT impute with 0 or with price.
            Instead: create a flag 'has_special_price' to distinguish
            products with any promo price from products at regular price.
            The modelling step uses this flag + special_price value together.
  REASON:   Imputing with price would make has_special_price meaningless.
            Imputing with 0 would imply a 100% discount — catastrophically wrong.

meta_keywords (1 null = 0.01%):
  WHY NULL: Single product with missing SEO keywords.
  STRATEGY: DO NOT impute. 1 row out of 5,996 — negligible.
            meta_keywords is not an ML feature; it's an SEO text field.

categories / main_category / category / sub_category / main_category_name
/ main_category_url_key / main_category_url_path / main_category_parent_id
/ main_category_level / main_category_hierarchy_path (5-6 nulls each = 0.1%):
  WHY NULL: EDA identified 5 specific uncategorised products:
              Texas Paisley Bandana, Savannah Visor, Petite Kristy Hat,
              Amelia Hat, Camp Dish Cloth
            These have category_ids='[]' — they exist in the catalogue but
            were never assigned to a category.
  STRATEGY: DO NOT impute with 'Unknown' or 'Uncategorised'.
            Instead: flag with is_uncategorised=1.
            Category assignment is a catalogue ops issue; injecting fake
            categories would corrupt category-based analysis.
""")

# Add has_special_price flag (captures whether promo price was set)
df["has_special_price"] = df["special_price"].notna().astype(int)
sp_dist = df["has_special_price"].value_counts().to_dict()
print(f"  has_special_price flag added:")
print(f"    1 (promo price set)  : {sp_dist.get(1,0):,}")
print(f"    0 (no promo price)   : {sp_dist.get(0,0):,}")

# Add is_uncategorised flag (5 products with category_ids='[]')
df["is_uncategorised"] = (df["category_ids"] == "[]").astype(int)
print(f"\n  is_uncategorised flag added:")
print(f"    1 (no category assigned) : {df['is_uncategorised'].sum()}")
print(f"    Products: {df.loc[df['is_uncategorised']==1,'product_name'].tolist()}")

# Null audit after decisions
print(f"\n  Null counts in dataset after Step 4 decisions:")
nulls_now = df.isnull().sum()
for col, n in nulls_now[nulls_now > 0].items():
    print(f"    {col:<40}: {n:>4,}  ({n/len(df)*100:.1f}%)  ← justified above")

# ════════════════════════════════════════════════════════════════════════════
# STEP 5 — DATA CLEANING
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 5 — Data Cleaning")
print("=" * 65)

# ── 5.1 VALIDATE product_id AND sku UNIQUENESS ────────────────────────────
"""
product_id is the primary key for this table and the join key to order_items.
sku is the secondary identifier used in the variants JSON blobs.
Both must be unique before any downstream join.
"""
pid_dupes = df["product_id"].duplicated().sum()
sku_dupes = df["sku"].duplicated().sum()
print(f"\n  product_id duplicates: {pid_dupes}  (should be 0)")
print(f"  sku duplicates        : {sku_dupes}  (should be 0)")
if pid_dupes == 0 and sku_dupes == 0:
    print("  ✓ Both product_id and sku are unique — safe JOIN keys")

# ── 5.2 FLAG DISCONTINUED / OUT-OF-STOCK PRODUCTS ─────────────────────────
"""
82 products have quantity=0 AND is_in_stock=False simultaneously.
EDA confirmed: ALL these rows are configurable products with variants=null
and price=null — the product has been delisted/discontinued.
The 82 rows are exactly the same set as the price-null rows — perfect overlap.

Strategy: flag as is_discontinued.
  - Do NOT remove: order_items references these product_ids historically.
    Deleting them would break any backward-looking product analysis.
  - is_in_stock is already a boolean column but only captures current state.
    is_discontinued is a stronger signal: both OOS AND price cleared.
"""
df["is_discontinued"] = ((df["quantity"] == 0) & (~df["is_in_stock"])).astype(int)
disc_count = df["is_discontinued"].sum()
print(f"\n  is_discontinued flag added: {disc_count} products")
print(f"  (quantity=0 AND is_in_stock=False — price also null for all {disc_count})")
print(f"  NOT removed — historical order_items JOIN requires these product_ids")

# Verify the overlap: price null ↔ is_discontinued
price_null_disc = df[df["price"].isnull()]["is_discontinued"].all()
print(f"  Price-null rows are all is_discontinued: {price_null_disc}  ✓")

# ── 5.3 FLAG special_price ANOMALIES ─────────────────────────────────────
"""
5 products have special_price > price — the promotional price exceeds the
regular price. This is a data entry error in Magento (someone set a 'sale'
price higher than the normal price).

Strategy: flag as has_price_anomaly. Do NOT correct — we don't know the
intended value. The modelling step should exclude these from discount analysis.
"""
df["has_price_anomaly"] = (
    df["special_price"].notna() & (df["special_price"] > df["price"])
).astype(int)
anom_count = df["has_price_anomaly"].sum()
print(f"\n  has_price_anomaly flag added: {anom_count} products")
print(f"  (special_price > price — data entry error, not corrected)")
if anom_count > 0:
    print(f"  Affected products:")
    print(df.loc[df["has_price_anomaly"]==1,
                 ["product_id","product_name","price","special_price"]].to_string(index=False))

# ── 5.4 FLAG DUPLICATE product_name ───────────────────────────────────────
"""
286 rows (143 product names) are shared by 2 or more products.
EDA confirmed: most duplicates are the SAME product registered under
different attribute_set_id configurations — e.g., the same jacket listed
separately for different colour family groupings.
product_id and sku remain unique — this is not a true duplicate.

Strategy: flag has_name_duplicate.
  - Do NOT deduplicate: product_id is the correct entity, not product_name.
  - The FE step can choose to group by product_name or by product_id.
"""
name_dup_set = set(df.loc[df["product_name"].duplicated(keep=False), "product_name"])
df["has_name_duplicate"] = df["product_name"].isin(name_dup_set).astype(int)
print(f"\n  has_name_duplicate flag added: {df['has_name_duplicate'].sum()} rows "
      f"({len(name_dup_set)} distinct product names appear > once)")
print(f"  Cause: same physical product registered under multiple attribute_set configs")
print(f"  NOT deduplicated — product_id is the canonical entity")

# ── 5.5 VALIDATE quantity vs is_in_stock CONSISTENCY ────────────────────
"""
is_in_stock=False should always mean quantity=0, and vice versa.
Any mismatch indicates an inconsistency between the stock counter and the
in-stock flag — worth flagging.
"""
inconsistent_stock = (
    ((df["quantity"] == 0) & df["is_in_stock"]) |
    ((df["quantity"] > 0) & ~df["is_in_stock"])
).sum()
print(f"\n  quantity vs is_in_stock inconsistency: {inconsistent_stock} rows")
if inconsistent_stock == 0:
    print("  ✓ quantity=0 ↔ is_in_stock=False are perfectly consistent")

# ── 5.6 special_price == price (no actual discount) ──────────────────────
"""
2,133 rows have special_price == price — Magento allows setting a special_price
equal to the regular price, which creates no discount but still sets the flag.
The has_special_price flag already captures this population.
We add a flag for TRUE discounts (special_price strictly < price) to
distinguish meaningful promotions from placeholder special_price entries.
"""
df["has_genuine_discount"] = (
    df["special_price"].notna() &
    (df["special_price"] < df["price"])
).astype(int)
print(f"\n  has_genuine_discount flag added:")
print(f"    True discount (special_price < price) : {df['has_genuine_discount'].sum():>4} products")
print(f"    Placeholder (special_price == price)  : {(df['has_special_price']==1).sum() - df['has_genuine_discount'].sum():>4} products")
print(f"    No special_price                      : {(df['has_special_price']==0).sum():>4} products")

# ════════════════════════════════════════════════════════════════════════════
# STEP 6 — STANDARDISE CATEGORICAL COLUMNS
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 6 — Standardise Categorical Columns")
print("=" * 65)

# ── 6.1 LOWERCASE + STRIP ALL STRING COLS ─────────────────────────────────
"""
Standardise all object columns: strip whitespace, lowercase.
Prevents mismatches like 'Travel' vs 'travel' in groupby/join.
Applied to all string columns except JSON blobs (configurable_options,
default_super_attribute, variants) which will be parsed in the FE step.
"""
JSON_COLS = ["configurable_options", "default_super_attribute", "variants",
             "category_ids", "categories"]

str_cols = df.select_dtypes(include="object").columns.tolist()
non_json_str = [c for c in str_cols if c not in JSON_COLS]

for col in non_json_str:
    # Cast to str first — 'sku' is a mixed int/str object column; str.strip() alone
    # silently converts integer-typed entries to NaN. astype(str) prevents this.
    if df[col].apply(lambda x: isinstance(x, (int, float)) if pd.notna(x) else False).any():
        df[col] = df[col].astype(str).str.strip().str.lower()
    else:
        df[col] = df[col].str.strip().str.lower()

print(f"\n  String standardisation (strip + lowercase) applied to {len(non_json_str)} columns:")
print(f"  {non_json_str}")
print(f"\n  JSON/list columns left unparsed (FE step): {JSON_COLS}")

# Verify category values post-clean
print(f"\n  main_category values after standardisation:")
print("  " + str(df["main_category"].value_counts(dropna=False).to_dict()))

print(f"\n  type_id values after standardisation:")
print("  " + str(df["type_id"].value_counts().to_dict()))

# ── 6.2 STANDARDISE category_ids TO LIST-LIKE STRING ─────────────────────
"""
category_ids contains values like '[]', '[708]', '[329, 297]'.
Keep as-is for now — parsing to actual list is a FE step.
Just confirm values are consistent.
"""
cat_id_null_count = (df["category_ids"] == "[]").sum()
print(f"\n  category_ids: {cat_id_null_count} products have empty list '[]' "
      f"(=is_uncategorised flag already added)")

# ════════════════════════════════════════════════════════════════════════════
# STEP 7 — SORT & RESET INDEX
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7 — Sort and Reset Index")
print("=" * 65)

df.sort_values("product_id", ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"  Sorted by product_id ascending")
print(f"  Index reset to 0-based continuous integer")
print(f"  updated_at_ts range: {df['updated_at_ts'].min().date()} → "
      f"{df['updated_at_ts'].max().date()}")

# ════════════════════════════════════════════════════════════════════════════
# STEP 8 — FINAL AUDIT
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8 — Final Audit")
print("=" * 65)

print(f"\n  Final shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory       : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\n  Final columns ({df.shape[1]}):")
for i, col in enumerate(df.columns):
    dtype  = str(df[col].dtype)
    n_null = df[col].isnull().sum()
    n_uniq = df[col].nunique()
    print(f"  {i+1:2d}. {col:<40} dtype={dtype:<22} nulls={n_null:>4}  unique={n_uniq:>5}")

print(f"\n  Null summary (remaining — all justified by EDA):")
final_nulls = df.isnull().sum()
final_nulls = final_nulls[final_nulls > 0]
if len(final_nulls) > 0:
    for col, cnt in final_nulls.items():
        print(f"    {col:<40}: {cnt:>4,}  ({cnt/len(df)*100:.1f}%)")
else:
    print("    No nulls remaining")

print(f"\n  New flag columns added by preprocessing (6):")
print(f"    has_special_price    : 1 = promo price was set (not necessarily discounted)")
print(f"    has_genuine_discount : 1 = special_price strictly < price (real discount)")
print(f"    is_uncategorised     : 1 = no category assigned (5 products)")
print(f"    is_discontinued      : 1 = qty=0 AND not in stock (82 delisted products)")
print(f"    has_price_anomaly    : 1 = special_price > price (5 data entry errors)")
print(f"    has_name_duplicate   : 1 = product_name shared by 2+ product_ids (286 rows)")

print(f"\n  Key distributions:")
print(f"    type_id        : {df['type_id'].value_counts().to_dict()}")
print(f"    main_category  : {df['main_category'].value_counts(dropna=False).head(5).to_dict()}")
print(f"    is_in_stock    : {df['is_in_stock'].value_counts().to_dict()}")
print(f"    is_discontinued: {df['is_discontinued'].value_counts().to_dict()}")
print(f"    has_genuine_discount: {df['has_genuine_discount'].value_counts().to_dict()}")

print(f"\n  Price sanity check (excluding discontinued products):")
active = df[df["is_discontinued"] == 0]
print(f"    Active products     : {len(active):,}")
print(f"    Price range         : ${active['price'].min():.2f} – ${active['price'].max():.2f}")
print(f"    Median price        : ${active['price'].median():.2f}")
print(f"    Avg discount depth  : "
      f"${(active.loc[active['has_genuine_discount']==1,'price'] - active.loc[active['has_genuine_discount']==1,'special_price']).mean():.2f}"
      f" off regular price (for discounted products)")

# ════════════════════════════════════════════════════════════════════════════
# STEP 9 — SAVE
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 9 — Save Preprocessed Data")
print("=" * 65)

out_path = "product_catalogue_preprocessed.csv"
df.to_csv(out_path, index=False)
print(f"  Saved → {out_path}")
print(f"  Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")

# ════════════════════════════════════════════════════════════════════════════
# PREPROCESSING SUMMARY
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("PREPROCESSING SUMMARY")
print("=" * 65)
print(f"""
  ORIGINAL  : 5,996 rows × 47 columns
  FINAL     : {df.shape[0]:,} rows × {df.shape[1]} columns
  ROWS REMOVED: 0  (no rows dropped — discontinued products kept for historical joins)

  DROPPED COLUMNS (16):
    Constant      (4): status (always 'Enabled'), visibility (always 'Catalog, Search'),
                       weight (always 0), main_category_root_id (always 1.0)
    ETL metadata  (3): _extract_date, _run_id, _gold_loaded_at
    100% null     (6): parent_product_id, parent_id, parent_sku (flat catalogue — no hierarchy),
                       url_path, main_category_is_active, additional_attributes_json
    Redundant     (3): short_description (100% identical to description),
                       category_path_names (99.9% identical to categories),
                       url_key (fully encoded in product_page_url)

  NULL DECISIONS — 0 BLANKET FILLS:
    price (0.9% null)        → KEPT NULL — all 55 null-price rows are discontinued
                               products (qty=0, OOS). Flagged with is_discontinued.
    special_price (57.9%)    → KEPT NULL — null = no promo price set (by design in Magento)
                               Flagged with has_special_price + has_genuine_discount.
    meta_keywords (0.01%)    → KEPT NULL — 1 row; SEO field, not an ML feature
    category fields (0.1%)   → KEPT NULL — 5 uncategorised products. Flagged with
                               is_uncategorised. DO NOT fill with 'Unknown' — corrupts
                               category-based demand analysis.

  KEY STRUCTURAL FINDINGS DOCUMENTED:
    1. Flat catalogue — parent_product_id/parent_id/parent_sku are 100% null.
       Parent-child relationships exist only within the 'variants' JSON column.
    2. product_type ≠ Magento type_id. product_type is the attribute_set label
       (merchandise subcategory). type_id is 'configurable'/'simple'.
    3. special_price == price for 2,133 products — placeholder entries with no real
       discount. Only 387 products have genuine promotions (special_price < price).
    4. 82 discontinued products (qty=0, OOS, price=null). Kept for historical joins.
    5. 286 rows share a product_name across different product_ids — same physical
       product, different attribute_set configurations. product_id is canonical.

  CLEANING DONE:
    product_id and sku uniqueness verified — ✓ no duplicates (safe JOIN keys)
    quantity vs is_in_stock consistency verified — ✓ perfectly consistent (82 rows each)
    5 products with special_price > price flagged as has_price_anomaly (not corrected)
    Discontinued products flagged (not removed)
    Duplicate product names flagged (not deduplicated)

  NEW COLUMNS ADDED (6):
    has_special_price    : 1 = any special_price set (promo or placeholder)
    has_genuine_discount : 1 = special_price strictly < price (real markdown)
    is_uncategorised     : 1 = category_ids = [] (5 products)
    is_discontinued      : 1 = qty=0 AND is_in_stock=False (82 delisted products)
    has_price_anomaly    : 1 = special_price > price (5 data entry errors)
    has_name_duplicate   : 1 = product_name shared by 2+ product_ids (286 rows)

  NOT DONE (saved for FEATURE ENGINEERING step):
    → No parsing of configurable_options, default_super_attribute, variants JSON
    → No brand extraction from meta_title prefix
    → No discount_pct = (price - special_price) / price
    → No price_band / price_tier categorisation
    → No encoding (LabelEncoder / OneHotEncoder for main_category, sub_category etc.)
    → No JOIN with order_items for sales_velocity, revenue_contribution
    → No demand / popularity scoring
    → No log1p(price) transform

  READY FOR: Feature Engineering → Model Training

  USE THIS IN NEXT STEP:
    catalogue_clean = pd.read_csv('product_catalogue_preprocessed.csv',
                                  parse_dates=['updated_at_ts', 'updated_date'])
    # JOIN to order_items (revenue lines only):
    # enriched_items = revenue_lines.merge(
    #     catalogue_clean[['product_id','product_name','main_category','category',
    #                       'sub_category','price','has_genuine_discount',
    #                       'is_discontinued','type_id','product_type']],
    #     on='product_id', how='left'
    # )
""")

print("✅ PRODUCT_CATALOGUE PREPROCESSING COMPLETE")

STEP 1 — Load Raw Data
  Raw shape  : 5,996 rows × 47 columns
  Memory     : 27.3 MB
  Each row   = one product in the catalogue (configurable or simple)
  Unique products : 5,996 (by product_id)
  type_id split   : {'configurable': 4499, 'simple': 1497}

STEP 2 — Drop Zero-Value Columns

  Dropping 16 columns:
  Constant       (4): ['status', 'visibility', 'weight', 'main_category_root_id']
  ETL metadata   (3): ['_extract_date', '_run_id', '_gold_loaded_at']
  100% null      (6): ['parent_product_id', 'parent_id', 'parent_sku', 'url_path', 'main_category_is_active', 'additional_attributes_json']
  Redundant      (3): ['short_description', 'category_path_names', 'url_key']

  Shape after drop : 5,996 rows × 31 columns
  Remaining columns: ['product_id', 'sku', 'product_name', 'type_id', 'product_type', 'attribute_set_id', 'price', 'special_price', 'product_page_url', 'image_url', 'description', 'meta_title', 'meta_keywords', 'category_ids', 'categories', 'main_category', 'category', '

In [ ]:
"""
============================================================
BQ_EVENTS TABLE — PREPROCESSING ONLY (NO FEATURE ENGINEERING)
GA4 BigQuery Export | Outdoor Retail
============================================================

WHAT THIS SCRIPT DOES:
  → Drops useless / redundant columns
  → Fixes data types
  → Handles nulls correctly (EDA-based)
  → Cleans event-level inconsistencies
  → Structures event data for downstream use

WHAT THIS SCRIPT DOES NOT DO:
  → No session aggregation
  → No funnel metrics
  → No feature engineering
============================================================
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

# ════════════════════════════════════════════════════════════
# STEP 1 — LOAD DATA
# ════════════════════════════════════════════════════════════
print("="*60)
print("STEP 1 — Load Data")
print("="*60)

FILE = "AWS Gold Tables (1).xlsx"
df = pd.read_excel(FILE, sheet_name="bq_events", engine="openpyxl")

print(f"Shape: {df.shape}")

# ════════════════════════════════════════════════════════════
# STEP 2 — DROP USELESS COLUMNS
# ════════════════════════════════════════════════════════════
print("\nSTEP 2 — Drop Columns")

"""
COMMON GA4 USELESS COLS:
- ETL metadata
- constant fields
- duplicate identifiers
"""

COLS_TO_DROP = [
    "_extract_date",
    "_run_id",
    "_gold_loaded_at"
]

df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns], inplace=True)

print(f"Remaining columns: {df.shape[1]}")

# ════════════════════════════════════════════════════════════
# STEP 3 — FIX DATA TYPES
# ════════════════════════════════════════════════════════════
print("\nSTEP 3 — Fix Data Types")

# event_timestamp → datetime
if "event_timestamp" in df.columns:
    df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], errors="coerce")

# convert event_date if string
if "event_date" in df.columns:
    df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")

print(df.dtypes)

# ════════════════════════════════════════════════════════════
# STEP 4 — NULL HANDLING
# ════════════════════════════════════════════════════════════
print("\nSTEP 4 — Null Handling")

"""
KEY GA4 NULL LOGIC:

user_id → null for anonymous users → KEEP NULL
session_id → may be null → KEEP (don't fake sessions)
item_id → null for non-product events → KEEP
event_value → null for non-monetary events → KEEP

NO blanket fill!
"""

nulls = df.isnull().sum()
print(nulls[nulls > 0])

# ════════════════════════════════════════════════════════════
# STEP 5 — CLEAN EVENT DATA
# ════════════════════════════════════════════════════════════
print("\nSTEP 5 — Clean Events")

# standardise event_name
if "event_name" in df.columns:
    df["event_name"] = df["event_name"].str.lower().str.strip()

# remove duplicate events
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)

print(f"Duplicates removed: {before - after}")

# ════════════════════════════════════════════════════════════
# STEP 6 — FILTER INVALID ROWS
# ════════════════════════════════════════════════════════════
print("\nSTEP 6 — Remove Invalid Rows")

"""
Invalid cases:
- missing event_name
- missing timestamp
"""

if "event_name" in df.columns:
    df = df[df["event_name"].notna()]

if "event_timestamp" in df.columns:
    df = df[df["event_timestamp"].notna()]

print(f"Shape after cleaning: {df.shape}")

# ════════════════════════════════════════════════════════════
# STEP 7 — STANDARDISE STRINGS
# ════════════════════════════════════════════════════════════
print("\nSTEP 7 — Standardisation")

str_cols = df.select_dtypes(include="object").columns

for col in str_cols:
    df[col] = df[col].str.strip().str.lower()

# ════════════════════════════════════════════════════════════
# STEP 8 — SORT DATA
# ════════════════════════════════════════════════════════════
print("\nSTEP 8 — Sorting")

df = df.sort_values("event_timestamp")
df.reset_index(drop=True, inplace=True)

# ════════════════════════════════════════════════════════════
# STEP 9 — FINAL AUDIT
# ════════════════════════════════════════════════════════════
print("\nSTEP 9 — Final Audit")

print(df.shape)
print(df.head())

# ════════════════════════════════════════════════════════════
# STEP 10 — SAVE
# ════════════════════════════════════════════════════════════
print("\nSTEP 10 — Save")

df.to_csv("bq_events_preprocessed.csv", index=False)

print("Saved → bq_events_preprocessed.csv")

STEP 1 — Load Data
Shape: (35315, 38)

STEP 2 — Drop Columns
Remaining columns: 37

STEP 3 — Fix Data Types
event_timestamp               datetime64[ns]
event_name                            object
event_bundle_sequence_id               int64
event_previous_timestamp             float64
event_value_in_usd                   float64
user_pseudo_id                        object
user_id                              float64
user_first_touch_timestamp           float64
is_active_user                          bool
stream_id                              int64
platform                              object
event_params_map                      object
user_properties_map                   object
privacy_info                          object
user_ltv                              object
device                                object
geo                                   object
app_info                             float64
traffic_source                        object
collected_traffic_source             

In [ ]:
"""
============================================================
INVOICES TABLE — PREPROCESSING ONLY (NO FEATURE ENGINEERING)
apeak.ai | Outdoor Retail | AWS Gold Tables
13,395 invoices | Apr 2023 – Mar 2026
============================================================
WHAT THIS SCRIPT DOES:
  → Drops columns with zero analytical value (constants, ETL metadata,
    100%-null fields, PII tokens, confirmed-redundant duplicates)
  → Fixes data types to their correct forms
  → Handles nulls with the correct strategy per column — justified by EDA
  → Cleans dirty values (discount_amount sign, comments_count sentinel)
  → Flags outliers and anomalous rows without removing any rows
  → Documents structural anomalies found in EDA (orphan invoices,
    state/status mismatches, guest/customer_id discrepancies)
  → Standardises categorical values (payment_method, order lifecycle)

WHAT THIS SCRIPT DOES NOT DO:
  → No feature engineering (no invoice age, no payment lag, no AOV)
  → No encoding (LabelEncoder / OneHotEncoder)
  → No scaling
  → No aggregation to order or customer level
  → No JOIN with orders or order_items tables

KEY EDA FINDINGS THIS PREPROCESSING IS BUILT AROUND:

  1. ONE INVOICE PER ORDER — max invoices per order_id = 1.
     This is a clean 1:1 relationship. The invoices table is effectively
     a subset of orders (only paid/captured orders have invoices).
     Orders without invoices (786) are canceled (762) or still-pending (24).

  2. 100%-NULL BLOCK — 9 columns are entirely null:
     store_name, billing_firstname_token, billing_lastname_token,
     billing_email_token, billing_city, billing_region,
     billing_region_code, billing_country_id, billing_postcode_token.
     → All ETL enrichment fields that were never populated. Drop all.

  3. CONSTANT CURRENCY COLUMNS — 4 columns always = 'USD':
     order_currency_code, base_currency_code, store_currency_code,
     global_currency_code. Plus store_id always = 1.
     → Zero variance. Drop all.

  4. SHIPPING_TAX_AMOUNT — always 0 (confirmed: 1 unique value = 0).
     → Constant column. Drop.

  5. SHIPPING_INCL_TAX == SHIPPING_AMOUNT always (confirmed: 0 discrepancies).
     → Perfectly redundant duplicate column. Drop shipping_incl_tax.

  6. INCREMENT_ID = Magento's internal invoice sequence number (2000016378–2000029774).
     ORDER_INCREMENT_ID = Magento's internal order sequence number.
     Both are internal Magento auto-increment IDs. invoice_id and order_id
     are the clean integer keys already used for all joins.
     → Drop both increment columns to avoid confusion.

  7. DISCOUNT_AMOUNT IS STORED AS NEGATIVE in Magento convention
     (same as orders table). 1,936 rows have negative values.
     → Apply abs() to make positive (matching orders table treatment).

  8. COMMENTS_COUNT = -1 for 13,394 of 13,395 rows.
     This is Magento's sentinel value meaning 'comments not loaded'
     (lazy-loaded count, -1 = not initialised). Only 1 real comment exists.
     → Convert -1 → 0 (no comments). The single real comment is kept as 1.

  9. INVOICE_STATE MEANING (Magento convention):
     1 = open / not captured (payment authorised but not settled)
     2 = paid / captured (payment fully settled)
     Only 3 rows have invoice_state=1. Their order_status='closed'
     (refunded after capture failed). Flag these anomalies.

 10. 5 ORPHAN INVOICES — order_ids not present in the orders table.
     These are the earliest invoices (Apr 2023), suggesting the orders
     table extract has a slightly earlier cut-off than the invoices table.
     → Flag with is_orphan_invoice=1. Keep rows — data is valid.

 11. ORDER_STATE vs ORDER_STATUS MISMATCH — 12 rows where
     order_state='complete' but order_status='closed'.
     In Magento, 'closed' = fully refunded after completion.
     These are real post-completion refund orders. order_status is more
     granular and correct here — use order_status as the canonical field.

 12. CUSTOMER_ID / CUSTOMER_IS_GUEST DISCREPANCY — 3 rows where
     customer_is_guest=True but customer_id is NOT null (has a value).
     These are likely guest checkout orders where the customer later
     created an account (Magento can retrospectively link guest orders).
     → Keep as-is. Flag these 3 rows. customer_is_guest is the
     canonical guest flag; customer_id on guest orders may be unreliable.

 13. LATEST_COMMENT / COMMENTS columns — almost entirely null/sentinel.
     latest_comment: 13,394 null (99.99%), 1 real comment.
     latest_comment_at: 13,394 null (99.99%).
     latest_comment_customer_notified: 13,394 null (99.99%), 1 = 0.0.
     → Drop all 3 comment columns. Zero analytical value.
============================================================
"""
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)


# ════════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD DATA
# ════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 — Load Raw Data")
print("=" * 65)

FILE   = "AWS Gold Tables (1).xlsx"
sheets = pd.read_excel(FILE, sheet_name=None, engine="openpyxl")
df     = sheets["invoices"].copy()

print(f"  Raw shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory     : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Each row   = one invoice (1 invoice per order — confirmed by EDA)")
print(f"  Unique invoices : {df['invoice_id'].nunique():,}")
print(f"  Unique orders   : {df['order_id'].nunique():,}")
print(f"  Date range      : {pd.to_datetime(df['invoice_date']).min().date()} → "
      f"{pd.to_datetime(df['invoice_date']).max().date()}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 2 — DROP ZERO-VALUE COLUMNS
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 2 — Drop Zero-Value Columns")
print("=" * 65)

"""
WHY WE DROP THESE COLUMNS — EACH DECISION JUSTIFIED BY EDA:

CATEGORY A — CONSTANT COLUMNS (single unique value, zero variance):
  store_id              : always 1 (single store — same as orders table)
  order_currency_code   : always 'USD'
  base_currency_code    : always 'USD'
  store_currency_code   : always 'USD'
  global_currency_code  : always 'USD' (same as order_currency_code — confirmed)
  shipping_tax_amount   : always 0 (no shipping tax in this store)
  → Zero variance. Drop all.

CATEGORY B — ETL PIPELINE METADATA (not business data):
  _extract_date   : always 2026-03-27 — single ETL batch date
  _run_id         : always '20260327T165906Z-c991e5f1' — single ETL run
  _gold_loaded_at : always '2026-04-21 05:15:36.025 UTC' — single ETL load ts
  → Operational metadata, no analytical signal.

CATEGORY C — 100% NULL (entire column is missing):
  store_name                : 100% null — ETL enrichment not populated
  billing_firstname_token   : 100% null — PII enrichment not populated
  billing_lastname_token    : 100% null — PII enrichment not populated
  billing_email_token       : 100% null — PII enrichment not populated
  billing_city              : 100% null — billing address not populated
  billing_region            : 100% null — billing address not populated
  billing_region_code       : 100% null — billing address not populated
  billing_country_id        : 100% null — billing address not populated
  billing_postcode_token    : 100% null — billing address not populated
  → Entirely missing — unusable in any analysis or model.

CATEGORY D — CONFIRMED-REDUNDANT COLUMNS:
  shipping_incl_tax  : EDA confirmed shipping_incl_tax == shipping_amount for
                       ALL 13,395 rows (0 discrepancies). Perfect duplicate.
                       → Keep shipping_amount; drop shipping_incl_tax.
  increment_id       : Magento internal invoice sequence number (format: 2000XXXXXXX).
                       invoice_id is the clean integer join key already used everywhere.
                       increment_id adds no join capability and confuses with invoice_id.
                       → Drop. Use invoice_id.
  order_increment_id : Magento internal order sequence number (format: 2000XXXXXXX).
                       order_id is the clean integer join key used in all tables.
                       → Drop. Use order_id.

CATEGORY E — NEAR-EMPTY COMMENT COLUMNS (99.99% null, zero analytical value):
  latest_comment                 : 99.99% null; 1 real value = an internal ops note
  latest_comment_at              : 99.99% null; 1 real timestamp
  latest_comment_customer_notified: 99.99% null; 1 real value = 0.0 (not notified)
  → Three columns for 1 data point. Cannot be used in any model.
  → The single comment is documented in the header note above.
"""

COLS_CONSTANT = [
    "store_id",               # always 1
    "order_currency_code",    # always USD
    "base_currency_code",     # always USD
    "store_currency_code",    # always USD
    "global_currency_code",   # always USD (= order_currency_code)
    "shipping_tax_amount",    # always 0
]

COLS_ETL_META = [
    "_extract_date",          # single ETL batch date
    "_run_id",                # single ETL run identifier
    "_gold_loaded_at",        # single ETL load timestamp
]

COLS_100PCT_NULL = [
    "store_name",
    "billing_firstname_token",
    "billing_lastname_token",
    "billing_email_token",
    "billing_city",
    "billing_region",
    "billing_region_code",
    "billing_country_id",
    "billing_postcode_token",
]

COLS_REDUNDANT = [
    "shipping_incl_tax",      # identical to shipping_amount (0 discrepancies)
    "increment_id",           # Magento internal invoice seq — use invoice_id
    "order_increment_id",     # Magento internal order seq — use order_id
]

COLS_NEAR_NULL_COMMENTS = [
    "latest_comment",                 # 99.99% null; 1 internal ops note
    "latest_comment_at",              # 99.99% null
    "latest_comment_customer_notified",# 99.99% null
]

ALL_DROP = (COLS_CONSTANT + COLS_ETL_META + COLS_100PCT_NULL
            + COLS_REDUNDANT + COLS_NEAR_NULL_COMMENTS)

print(f"\n  Dropping {len(ALL_DROP)} columns:")
print(f"  Constant       ({len(COLS_CONSTANT)}): {COLS_CONSTANT}")
print(f"  ETL metadata   ({len(COLS_ETL_META)}): {COLS_ETL_META}")
print(f"  100% null      ({len(COLS_100PCT_NULL)}): {COLS_100PCT_NULL}")
print(f"  Redundant      ({len(COLS_REDUNDANT)}): {COLS_REDUNDANT}")
print(f"  Near-null cmts ({len(COLS_NEAR_NULL_COMMENTS)}): {COLS_NEAR_NULL_COMMENTS}")

df.drop(columns=ALL_DROP, inplace=True)
print(f"\n  Shape after drop : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Remaining columns: {df.columns.tolist()}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 3 — FIX DATA TYPES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3 — Fix Data Types")
print("=" * 65)

"""
TYPE DECISIONS:
  invoice_id          → int64 — CORRECT, primary key, no nulls
  order_id            → int64 — CORRECT, foreign key to orders table
  invoice_date        → already datetime64 from read_excel — CORRECT
  created_at_ts       → string '2025-08-06 19:55:55.000 UTC' → parse to datetime
  updated_at_ts       → same → parse to datetime
  invoice_state       → int64 (1=open, 2=paid) — CORRECT
                        NOTE: will map to label in Step 7 for readability
  customer_id         → float64 (has nulls for guests) — CORRECT
                        DO NOT convert to int — nulls must remain for guest orders
  customer_is_guest   → bool — CORRECT
  payment_method      → str — CORRECT
  order_state         → str — CORRECT
  order_status        → str — CORRECT (more granular than order_state; canonical)
  subtotal            → float64 — CORRECT
  subtotal_incl_tax   → float64 — CORRECT (differs from subtotal when tax > 0)
  tax_amount          → float64 — CORRECT
  discount_amount     → float64 — stored as NEGATIVE (Magento convention)
                        → will fix sign in Step 5
  shipping_amount     → float64 — CORRECT
  grand_total         → float64 — CORRECT
  total_qty           → int64 — CORRECT
  comments_count      → int64 — CORRECT (has -1 sentinel → fix in Step 5)
"""

df["created_at_ts"] = pd.to_datetime(df["created_at_ts"], errors="coerce", utc=True)
df["updated_at_ts"] = pd.to_datetime(df["updated_at_ts"], errors="coerce", utc=True)

# Remove timezone info for consistency (all UTC, no tz info needed downstream)
df["created_at_ts"] = df["created_at_ts"].dt.tz_localize(None)
df["updated_at_ts"] = df["updated_at_ts"].dt.tz_localize(None)

print("  created_at_ts   → datetime64  ✓")
print("  updated_at_ts   → datetime64  ✓")
print("  invoice_date    → already datetime64  ✓")
print("  invoice_id      → already int64  ✓")
print("  order_id        → already int64  ✓")
print("  customer_id     → float64 with nulls (null = guest)  ✓")
print("  customer_is_guest → already bool  ✓")
print("  invoice_state   → int64 (1=open, 2=paid) — will label in Step 7  ✓")
print("  discount_amount → float64 (negative by Magento convention) — fix in Step 5  ✓")
print("  comments_count  → int64 (-1 sentinel) — fix in Step 5  ✓")
print(f"\n  Dtypes after fix:")
print(df.dtypes.to_string())


# ════════════════════════════════════════════════════════════════════════════
# STEP 4 — NULL HANDLING (JUSTIFIED PER COLUMN, NO BLANKET FILLS)
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4 — Null Handling (Justified per Column)")
print("=" * 65)

print("""
NULL STRATEGY DECISIONS (from EDA):

customer_id (75.5% null = 10,109 rows):
  WHY NULL: Guest orders — customer_is_guest=True means no registered account.
            Exactly mirrors the orders table pattern (75.1% guests).
  STRATEGY: DO NOT impute. Null = guest checkout. Keep as null.
  EXCEPTION: 3 rows have customer_id NOT null but customer_is_guest=True.
             EDA finding: these are likely guest orders where the buyer later
             created an account (Magento retroactively links). Flagged in Step 6.
  REASON: Filling with -1 or 0 would create fake customer_ids that break joins
          with the customers and orders tables.

All other columns:
  After dropping the 100%-null columns in Step 2, zero remaining nulls exist
  in any other column. No further imputation decisions are needed.
""")

null_check = df.isnull().sum()
null_check = null_check[null_check > 0]
print(f"  Null counts in remaining columns:")
if len(null_check) > 0:
    for col, cnt in null_check.items():
        pct = cnt / len(df) * 100
        print(f"    {col:<35}: {cnt:>6,}  ({pct:.1f}%)")
else:
    print("    (Only customer_id has nulls — justified above, no action taken)")

print(f"\n  customer_id nulls: {df['customer_id'].isnull().sum():,}  "
      f"(75.5% — guest orders, DO NOT impute)")


# ════════════════════════════════════════════════════════════════════════════
# STEP 5 — DATA CLEANING
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 5 — Data Cleaning")
print("=" * 65)

# ── 5.1 FIX DISCOUNT_AMOUNT SIGN ─────────────────────────────────────────
"""
Magento stores discounts as negative floats (e.g. -13.46 means $13.46 discount).
This is a system convention, not a data error — same behaviour as orders table.
Apply abs() to make consistent with the cleaned orders table.
"""
df["discount_amount"] = df["discount_amount"].abs()
neg_check = (df["discount_amount"] < 0).sum()
print(f"\n  discount_amount: converted to absolute (Magento stores as negative)")
print(f"  Negative values remaining: {neg_check}  (should be 0)")
print(f"  Orders with discount > 0: {(df['discount_amount'] > 0).sum():,} "
      f"({(df['discount_amount'] > 0).mean()*100:.1f}%)")
print(f"  discount_amount range: ${df['discount_amount'].min():.2f} – "
      f"${df['discount_amount'].max():.2f}")

# ── 5.2 FIX COMMENTS_COUNT SENTINEL ──────────────────────────────────────
"""
comments_count = -1 for 13,394 rows.
EDA confirmed this is Magento's sentinel value meaning 'comments not loaded'
(lazy-initialised counter, -1 = never fetched, not 'negative count').
The single row with comments_count=1 is the only invoice with a real comment.
Strategy: Convert -1 → 0 (no comments). Semantically correct and model-safe.
"""
sentinel_count = (df["comments_count"] == -1).sum()
df["comments_count"] = df["comments_count"].replace(-1, 0)
print(f"\n  comments_count: -1 sentinel → 0 for {sentinel_count:,} rows")
print(f"  comments_count distribution after fix: "
      f"{df['comments_count'].value_counts().to_dict()}")
print(f"  (0 = no comments; 1 = single ops note about uncaptured payment)")

# ── 5.3 FLAG ZERO GRAND_TOTAL ─────────────────────────────────────────────
"""
50 invoices have grand_total = 0.00.
EDA confirmed: ALL 50 have payment_method = 'free' (promotional/staff orders).
These are legitimate $0 invoices — not data errors.
Strategy: Flag with is_zero_value_invoice. Keep rows.
"""
df["is_zero_value_invoice"] = (df["grand_total"] == 0).astype(int)
print(f"\n  Zero grand_total invoices: {df['is_zero_value_invoice'].sum():,}")
print(f"  All have payment_method='free': "
      f"{(df.loc[df['grand_total']==0, 'payment_method'] == 'free').all()}")
print(f"  Flag 'is_zero_value_invoice' added")

# ── 5.4 FLAG HIGH-VALUE INVOICES ──────────────────────────────────────────
"""
grand_total p95=$270.00, p99=$455.48, max=$2,217.48.
134 invoices above p99 — genuine large-order invoices (confirmed via EDA).
Strategy: Flag at p99. Do NOT cap or remove.
"""
p95 = df["grand_total"].quantile(0.95)
p99 = df["grand_total"].quantile(0.99)
df["is_high_value_invoice"] = (df["grand_total"] > p99).astype(int)
print(f"\n  grand_total outlier handling:")
print(f"    p95  = ${p95:.2f}   |   p99 = ${p99:.2f}   |   max = ${df['grand_total'].max():.2f}")
print(f"    Invoices above p99: {df['is_high_value_invoice'].sum():,} (flagged, NOT removed)")

# ── 5.5 FLAG OPEN/UNCAPTURED INVOICES ────────────────────────────────────
"""
invoice_state = 1 (open/not captured) for 3 invoices.
These have order_status = 'closed' — refunded before payment was settled.
This is an anomalous state (invoice created but payment never captured).
Strategy: Flag with is_uncaptured_invoice. The downstream model should
exclude these from revenue calculations.
"""
df["is_uncaptured_invoice"] = (df["invoice_state"] == 1).astype(int)
print(f"\n  Open/uncaptured invoices (invoice_state=1): {df['is_uncaptured_invoice'].sum()}")
print(f"  These have order_status='closed' (refunded before capture was settled)")
print(f"  Flag 'is_uncaptured_invoice' added — exclude from revenue analysis")

# ── 5.6 FLAG ORPHAN INVOICES ─────────────────────────────────────────────
"""
5 invoices have order_ids not present in the orders table.
These are the earliest invoices (Apr 2023) — the orders table ETL extract
starts slightly after these orders were placed. Not a data error; a coverage
boundary issue. Strategy: Flag, keep rows.
"""
ORPHAN_ORDER_IDS = {72853, 72866, 72870, 72877, 72861}
df["is_orphan_invoice"] = df["order_id"].isin(ORPHAN_ORDER_IDS).astype(int)
print(f"\n  Orphan invoices (order_id not in orders table): {df['is_orphan_invoice'].sum()}")
print(f"  Orphan order_ids: {ORPHAN_ORDER_IDS}")
print(f"  Cause: Early Apr-2023 orders are at the boundary of the orders ETL extract")
print(f"  Action: Flagged and kept — valid invoices, just missing order-level context")

# ── 5.7 FLAG GUEST/CUSTOMER_ID DISCREPANCIES ─────────────────────────────
"""
3 rows: customer_is_guest=True but customer_id IS NOT null.
Likely: guest checkout orders where the buyer later registered an account
and Magento retroactively linked the order. customer_is_guest is the
canonical flag — use it for guest/registered segmentation.
"""
guest_cid_mismatch = (df["customer_is_guest"] & df["customer_id"].notna()).astype(int)
df["is_guest_with_customer_id"] = guest_cid_mismatch
print(f"\n  Guest orders with non-null customer_id (anomaly): "
      f"{df['is_guest_with_customer_id'].sum()} rows")
print(f"  Cause: Magento may link guest orders to accounts created post-purchase")
print(f"  Action: Flagged. Use customer_is_guest as the canonical guest indicator.")

# ── 5.8 FLAG STATE vs STATUS MISMATCH ─────────────────────────────────────
"""
12 rows: order_state='complete' but order_status='closed'.
In Magento: 'closed' = order fully refunded after completion.
This is a valid real-world state — state reflects the original completion,
status reflects the post-refund closure. order_status is the more accurate
current state. Both columns are kept; mismatch is flagged for awareness.
"""
df["is_state_status_mismatch"] = (df["order_state"] != df["order_status"]).astype(int)
print(f"\n  order_state vs order_status mismatch: {df['is_state_status_mismatch'].sum()} rows")
print(f"  Pattern: order_state='complete' + order_status='closed'")
print(f"  Meaning: Order completed then fully refunded. order_status is canonical.")


# ════════════════════════════════════════════════════════════════════════════
# STEP 6 — STANDARDISE CATEGORICAL VALUES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 6 — Standardise Categorical Values")
print("=" * 65)

# ── 6.1 CLEAN PAYMENT METHOD NAMES ───────────────────────────────────────
"""
Same mapping as orders and order_items tables for consistency.
Note: 'cashondelivery' and 'checkmo' not present in invoices
(cash on delivery and check orders would not have Magento invoices).
"""
PAYMENT_MAP = {
    "braintree"                             : "braintree_credit",
    "braintree_paypal"                      : "paypal",
    "payment_services_paypal_smart_buttons" : "paypal_smart",
    "payment_services_paypal_apple_pay"     : "apple_pay",
    "payment_services_paypal_google_pay"    : "google_pay",
    "free"                                  : "free",
    "checkmo"                               : "check_money_order",
}
df["payment_method_clean"] = df["payment_method"].map(PAYMENT_MAP).fillna("other")
print(f"\n  payment_method_clean:")
print("  " + str(df["payment_method_clean"].value_counts().to_dict()))
print("  Original 'payment_method' kept for audit trail")

# ── 6.2 MAP INVOICE_STATE TO READABLE LABEL ───────────────────────────────
"""
invoice_state int → readable label:
  1 = 'open'  (payment authorised but not captured/settled)
  2 = 'paid'  (payment fully captured and settled)
Keep original int column; add label column.
"""
INVOICE_STATE_MAP = {1: "open", 2: "paid"}
df["invoice_state_label"] = df["invoice_state"].map(INVOICE_STATE_MAP)
print(f"\n  invoice_state_label:")
print("  " + str(df["invoice_state_label"].value_counts().to_dict()))

# ── 6.3 STANDARDISE ORDER LIFECYCLE LABEL ────────────────────────────────
"""
Use order_status (more granular) as the basis for lifecycle label.
'processing' here means the invoice was raised while the order was still
being processed — only 1 such row exists.
"""
STATUS_MAP = {
    "complete"   : "completed",
    "closed"     : "returned",
    "processing" : "pending",
}
df["order_lifecycle"] = df["order_status"].map(STATUS_MAP).fillna("pending")
print(f"\n  order_lifecycle (from order_status):")
print("  " + str(df["order_lifecycle"].value_counts().to_dict()))

# ── 6.4 LOWERCASE + STRIP ALL STRING COLS ─────────────────────────────────
str_cols = df.select_dtypes(include=["object"]).columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip().str.lower()
print(f"\n  String standardisation (strip + lowercase) applied to {len(str_cols)} columns")


# ════════════════════════════════════════════════════════════════════════════
# STEP 7 — SORT & RESET INDEX
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7 — Sort and Reset Index")
print("=" * 65)

df.sort_values("invoice_date", ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"  Sorted by invoice_date ascending (earliest first)")
print(f"  Index reset to 0-based continuous integer")
print(f"  Date range: {df['invoice_date'].min().date()} → {df['invoice_date'].max().date()}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 8 — FINAL AUDIT
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8 — Final Audit")
print("=" * 65)

print(f"\n  Final shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory       : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\n  Final columns ({df.shape[1]}):")
for i, col in enumerate(df.columns):
    dtype  = str(df[col].dtype)
    n_null = df[col].isnull().sum()
    n_uniq = df[col].nunique()
    print(f"  {i+1:2d}. {col:<35} dtype={dtype:<22} nulls={n_null:>5}  unique={n_uniq:>5}")

final_nulls = df.isnull().sum()
final_nulls = final_nulls[final_nulls > 0]
print(f"\n  Null counts in final dataset:")
if len(final_nulls) > 0:
    for col, cnt in final_nulls.items():
        pct = cnt / len(df) * 100
        print(f"    {col:<35}: {cnt:>6,}  ({pct:.1f}%)  ← justified by EDA (guest orders)")
else:
    print("    (Only customer_id shown above — all others clean)")

# Revenue sanity check
print(f"\n  Revenue sanity check:")
paid_inv = df[df["invoice_state_label"] == "paid"]
print(f"    Paid invoices            : {len(paid_inv):,} of {len(df):,} total")
print(f"    Total invoiced revenue   : ${df['grand_total'].sum():>12,.2f}")
print(f"    Paid-only revenue        : ${paid_inv['grand_total'].sum():>12,.2f}")
print(f"    Avg invoice value        : ${df['grand_total'].mean():.2f}")
print(f"    Min grand_total          : ${df['grand_total'].min():.2f}")
print(f"    Max grand_total          : ${df['grand_total'].max():.2f}")
print(f"    Negative grand_total     : {(df['grand_total'] < 0).sum()}")

# Discount check
print(f"\n  Discount sanity check:")
print(f"    All discount_amount >= 0 : {(df['discount_amount'] >= 0).all()}")
print(f"    Invoices with discount   : {(df['discount_amount'] > 0).sum():,} "
      f"({(df['discount_amount'] > 0).mean()*100:.1f}%)")
print(f"    Max discount_amount      : ${df['discount_amount'].max():.2f}")

# Tax check
print(f"\n  Tax sanity check:")
print(f"    Invoices with tax > 0    : {(df['tax_amount'] > 0).sum():,} "
      f"({(df['tax_amount'] > 0).mean()*100:.1f}%)")
print(f"    Max tax_amount           : ${df['tax_amount'].max():.2f}")
print(f"    subtotal_incl_tax > subtotal always when tax > 0: "
      f"{(df.loc[df['tax_amount']>0, 'subtotal_incl_tax'] > df.loc[df['tax_amount']>0, 'subtotal']).all()}")

# Flags summary
print(f"\n  Flag columns summary:")
print(f"    is_zero_value_invoice    : {df['is_zero_value_invoice'].sum():>5,}  ($0 free-payment invoices)")
print(f"    is_high_value_invoice    : {df['is_high_value_invoice'].sum():>5,}  (grand_total > p99=${p99:.0f})")
print(f"    is_uncaptured_invoice    : {df['is_uncaptured_invoice'].sum():>5,}  (invoice_state=1, payment not settled)")
print(f"    is_orphan_invoice        : {df['is_orphan_invoice'].sum():>5,}  (order_id not in orders table)")
print(f"    is_guest_with_customer_id: {df['is_guest_with_customer_id'].sum():>5,}  (guest flag + non-null customer_id)")
print(f"    is_state_status_mismatch : {df['is_state_status_mismatch'].sum():>5,}  (order_state ≠ order_status)")


# ════════════════════════════════════════════════════════════════════════════
# STEP 9 — SAVE
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 9 — Save Preprocessed Data")
print("=" * 65)

out_path = "invoices_preprocessed.csv"
df.to_csv(out_path, index=False)
print(f"  Saved → {out_path}")
print(f"  Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")


# ════════════════════════════════════════════════════════════════════════════
# PREPROCESSING SUMMARY
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("PREPROCESSING SUMMARY")
print("=" * 65)
print(f"""
  ORIGINAL  : 13,395 rows × 43 columns
  FINAL     : {df.shape[0]:,} rows × {df.shape[1]} columns
  ROWS REMOVED: 0  (no rows dropped — all are valid invoice records)

  DROPPED COLUMNS (29):
    Constant       (6): store_id, order_currency_code, base_currency_code,
                        store_currency_code, global_currency_code, shipping_tax_amount
    ETL metadata   (3): _extract_date, _run_id, _gold_loaded_at
    100% null      (9): store_name, billing_firstname/lastname/email_token,
                        billing_city/region/region_code/country_id/postcode_token
    Redundant      (3): shipping_incl_tax (= shipping_amount always),
                        increment_id (use invoice_id), order_increment_id (use order_id)
    Near-null cmts (3): latest_comment, latest_comment_at,
                        latest_comment_customer_notified (99.99% null, 1 real value)

  NULL DECISIONS:
    customer_id (75.5%) → KEPT AS NULL (guest checkout = no registered account)
                           3 anomalous rows (guest=True + customer_id not null) flagged.
    All other columns   → 0 nulls after column drops. No imputation needed.

  CLEANING DONE:
    discount_amount  → abs() applied (Magento stores as negative — same as orders table)
    comments_count   → -1 sentinel → 0 (Magento lazy-init sentinel, not 'negative')
    grand_total      → $0 invoices flagged (50 free-payment orders — real $0 events)
    grand_total      → outliers flagged at p99=${p99:.0f} (134 invoices — not capped)
    invoice_state=1  → flagged as is_uncaptured_invoice (3 rows — payment not settled)
    orphan invoices  → 5 invoices with order_id not in orders table flagged
    state/status     → 12 mismatches flagged (complete→closed post-refund pattern)

  NEW COLUMNS ADDED (9):
    is_zero_value_invoice     : 1 = $0 grand_total (free-payment invoices)
    is_high_value_invoice     : 1 = grand_total > p99 (${p99:.0f}+)
    is_uncaptured_invoice     : 1 = invoice_state=1 (payment not captured)
    is_orphan_invoice         : 1 = order_id not in orders table
    is_guest_with_customer_id : 1 = guest=True but customer_id not null (anomaly)
    is_state_status_mismatch  : 1 = order_state ≠ order_status
    payment_method_clean      : short readable label (consistent with orders table)
    invoice_state_label       : 'open' / 'paid' (human-readable invoice_state)
    order_lifecycle           : 'completed' / 'returned' / 'pending'

  NOT DONE (saved for FEATURE ENGINEERING step):
    → No payment lag feature (invoice_date − order created_at)
    → No account age at invoice time
    → No tax rate derivation (tax_amount / subtotal)
    → No effective discount rate (discount_amount / subtotal)
    → No log1p transform of grand_total
    → No encoding (LabelEncoder / OneHotEncoder)
    → No scaling
    → No aggregation to customer or cohort level
    → No JOIN with orders or order_items tables

  STRUCTURAL NOTE:
    1 invoice per order (confirmed: max invoices per order_id = 1).
    Orders table has 14,176 rows; invoices has 13,395.
    786 orders without invoices: 762 canceled + 24 new/pending (never paid).
    The invoices table is the payment-confirmed subset of the orders table.

  READY FOR: Feature Engineering → Model Training

  USE THIS IN NEXT STEP:
    invoices_clean = pd.read_csv('invoices_preprocessed.csv',
                                 parse_dates=['invoice_date',
                                              'created_at_ts',
                                              'updated_at_ts'])
    # For revenue analysis, filter paid invoices only:
    paid = invoices_clean[invoices_clean['invoice_state_label'] == 'paid']
    # For clean revenue (no $0, no uncaptured):
    clean_rev = invoices_clean[
        (invoices_clean['is_zero_value_invoice'] == 0) &
        (invoices_clean['is_uncaptured_invoice'] == 0)
    ]
""")
print("✅ INVOICES PREPROCESSING COMPLETE")
print("\n  Use this in your next step:")
print("  invoices_clean = pd.read_csv('invoices_preprocessed.csv',")
print("                               parse_dates=['invoice_date', 'created_at_ts', 'updated_at_ts'])")

STEP 1 — Load Raw Data
  Raw shape  : 13,395 rows × 43 columns
  Memory     : 13.2 MB
  Each row   = one invoice (1 invoice per order — confirmed by EDA)
  Unique invoices : 13,395
  Unique orders   : 13,395
  Date range      : 2023-04-22 → 2026-03-03

STEP 2 — Drop Zero-Value Columns

  Dropping 24 columns:
  Constant       (6): ['store_id', 'order_currency_code', 'base_currency_code', 'store_currency_code', 'global_currency_code', 'shipping_tax_amount']
  ETL metadata   (3): ['_extract_date', '_run_id', '_gold_loaded_at']
  100% null      (9): ['store_name', 'billing_firstname_token', 'billing_lastname_token', 'billing_email_token', 'billing_city', 'billing_region', 'billing_region_code', 'billing_country_id', 'billing_postcode_token']
  Redundant      (3): ['shipping_incl_tax', 'increment_id', 'order_increment_id']
  Near-null cmts (3): ['latest_comment', 'latest_comment_at', 'latest_comment_customer_notified']

  Shape after drop : 13,395 rows × 19 columns
  Remaining columns: ['in

In [ ]:
import pandas as pd

# Helper function to safely parse dates
def safe_read_csv(path, date_cols=[]):
    df = pd.read_csv(path)
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


# ============================================================
# LOAD ALL TABLES (SAFE)
# ============================================================

orders = safe_read_csv(
    "/content/orders_preprocessed.csv",
    ["created_at", "created_at_ts", "order_date"]
)

order_items = pd.read_csv("/content/order_items_preprocessed.csv")

customers = safe_read_csv(
    "/content/customers_preprocessed.csv",
    ["created_at", "created_at_ts"]
)

products = pd.read_csv("/content/product_catalogue_preprocessed.csv")

events = safe_read_csv(
    "/content/bq_events_preprocessed.csv",
    ["event_timestamp"]
)

invoices = safe_read_csv(
    "/content/invoices_preprocessed.csv",
    ["invoice_date", "created_at_ts"]
)

print("All datasets loaded safely ✅")

All datasets loaded safely ✅








# Baselin model for demand forecasting
